# Notebook A — Data Preparation, SOH Target, RQ1 Analysis & Handoff

**Project:** Cross-Battery State-of-Health Estimation for UAV-grade Lithium-Ion Cells under Randomized Loading  
**Dataset:** NASA PCoE Randomized Battery Usage (`RW1`–`RW28`)  
**Group:** Nhóm 7  
**Role of this notebook:** prepare the data, construct the SOH target, audit data quality, engineer leakage-safe rolling features, answer **RQ1**, and hand off a reproducible feature table to Notebook B.

### What this notebook produces

The main handoff is `data/processed/feat.parquet` together with `data/processed/manifest.json`. Supporting evidence is written to `report/` and SQL is saved to `sql/queries.sql`.

A successful clean run should end with approximately:

- **548,010 feature rows**
- **17 columns**
- **28 battery cells**
- all final validation tests passing
- the manual 50-step rolling-window check passing
- the Notebook B handoff simulation passing

> **Reading guide:** each section first explains *what is being done*, *why it is needed*, and *how to interpret the result*. Code cells then create the corresponding evidence.


## Research integrity and AI-assisted audit trail

AI was used as a coding and review assistant, but important decisions were checked against the actual data rather than accepted automatically.

Three examples of **Human Delta** in this notebook are especially important:

1. A broad text match for `"reference discharge"` also captured rows such as `"rest post reference discharge"`. The production target therefore uses an exact rule: `type == "D"` and cleaned comment exactly equal to `"reference discharge"`.
2. Temperatures around `-4099 °C` were found in the raw records. These were treated as invalid sensor/sentinel values. The **temperature value** is marked missing, but the entire operational row is retained.
3. The SQL rolling features were independently recomputed by hand for a selected row. The manual and SQL 50-step results must match before the feature table is handed to Notebook B.

The individual AI Audit Log should record prompts that materially changed the analysis, together with how the result was verified or corrected.


## 0. Environment and reproducibility

This project is intended to run from the project root, for example:

```text
E:\PJ_ADY
```

The notebook uses a project-local Python environment:

```powershell
python -m venv .venv
.\.venv\Scripts\Activate.ps1
pip install -r requirements.txt
```

In VS Code, select the `.venv` Python kernel before running the notebook.

The current reproducible environment is based on **Python 3.10.11**. Package versions are pinned in `requirements.txt`. Notebook A deliberately keeps only the libraries it actually needs; model-specific packages belong to Notebook B when required.

> The optional `ydata-profiling` cell may display a deprecation warning. In the verified environment this is a warning, not a failure of the analysis.


In [1]:
# Environment check and shared imports for the whole notebook
import pandas as pd, numpy as np, duckdb, matplotlib
import matplotlib.pyplot as plt, pathlib, warnings
warnings.filterwarnings('ignore')
for d in ['data/raw', 'data/processed', 'notebooks', 'sql', 'src', 'report']: pathlib.Path(d).mkdir(parents=True, exist_ok=True)
np.random.seed(42); pd.set_option('display.width', 160); pd.set_option('display.max_columns', 40)
print('pandas', pd.__version__, '| duckdb', duckdb.__version__)

pandas 2.3.3 | duckdb 1.5.5


## Notebook A workflow

| Section | Purpose | Main evidence |
|---|---|---|
| 1. Raw data | Load the NASA MATLAB files and verify the dataset structure | raw audit, `profile.html` |
| 2. SOH target & cleaning | Identify true reference discharges, construct SOH, and document data quality | cleaning log, SOH audits |
| 3. Temperature quality | Detect invalid temperature sentinel values without discarding otherwise useful rows | quality and missing-value tables |
| 4. SQL & features | Summarize RQ1 evidence and create 50-step history features using past/current data only | Q1–Q4, `queries.sql`, `feat.parquet` |
| 5. RQ1 analysis | Quantify degradation versus throughput and examine load-history associations | RQ1A–RQ1D tables |
| 6. RQ1 figures | Communicate degradation, cross-cell heterogeneity, and confounding visually | seven `fig_rq1_*.png` files |
| 7. Validation & handoff | Prove the feature table is internally consistent and reproducible | tests, hashes, `manifest.json` |

Notebook B will use the handoff table for cross-cell model comparison, uncertainty/conformal analysis, EOL evaluation, ablation, and explainability.


## 1. Research scope and raw NASA data

### Research questions

**RQ1 — Degradation pattern.** How does SOH change with cumulative operational throughput, and how strongly are recent load-history summaries associated with SOH?

**RQ2 — Cross-battery prediction.** Which predictive models generalize best to battery cells that were not used for training?  
*Handled in Notebook B.*

**RQ3 — Reliability and decision support.** How reliable are uncertainty intervals and EOL decisions on unseen cells, and which features contribute to the predictions?  
*Handled in Notebook B.*

### Target used in this project

The target is **State of Health (SOH)** derived from periodic reference-discharge capacity measurements. Operational rows between reference tests are assigned the capacity measured at the **next true reference discharge** for the same cell.

Conceptually:

\[
SOH = \frac{\text{next reference-discharge capacity}}{\text{initial reference capacity for that cell}}
\]

The target is therefore tied to measured reference tests rather than inferred directly from the instantaneous random-load step.

### Raw-data expectation

The NASA randomized-usage files are searched recursively under `data/raw/`. A valid load should find all 28 cells (`RW1`–`RW28`). The raw step table contains more than one million charge, discharge, and rest records.


In [2]:
# Step 1b: NASA PCoE Randomized Battery Usage Data Set (Bole et al., 2014): 18650 cells cycled with random current profiles, MATLAB files per cell
from scipy.io import loadmat; import pandas as pd, numpy as np, glob, os
meas, steps = [], []
for f in sorted(glob.glob('data/raw/rw_battery/**/RW*.mat', recursive=True)):
    cell = os.path.basename(f)[:-4]; m = loadmat(f, simplify_cells=True)['data']['step']
    for k, s in enumerate(m):
        v, i, T, t = np.atleast_1d(s['voltage']), np.atleast_1d(s['current']), np.atleast_1d(s['temperature']), np.atleast_1d(s['relativeTime'])
        steps.append({'cell': cell, 'step': k, 'type': s['type'], 'comment': str(s.get('comment', '')), 'n': len(v), 'dur_s': float(t[-1]) if len(t) else 0.0,
                      'v_start': float(v[0]) if len(v) else np.nan, 'v_end': float(v[-1]) if len(v) else np.nan, 'i_mean': float(np.mean(i)) if len(i) else np.nan, 't_max': float(np.max(T)) if len(T) else np.nan,
                      'ah': float(np.trapz(np.abs(i), t) / 3600) if len(t) > 1 else 0.0})
steps = pd.DataFrame(steps); print(steps.shape, steps.cell.nunique(), steps.type.value_counts().head().to_dict())

(1117894, 11) 28 {'R': 561172, 'D': 423810, 'C': 132912}


### 1.1 Automated profile

A 20,000-row sample is profiled with `ydata-profiling` and saved as `report/profile.html`.

This is a **screening tool**, not the final analysis. It is used to quickly spot unusual distributions, missingness, and candidate data-quality issues that are then checked explicitly in later cells.


In [3]:
from ydata_profiling import ProfileReport
sample = steps.sample(min(20000, len(steps)), random_state=42)
ProfileReport(sample, minimal=True).to_file('report/profile.html')
steps.describe(include='all').T.to_csv('report/table_describe_raw.csv')

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 11/11 [00:00<00:00, 511.46it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

## 2. Constructing the SOH target

This is the most important data-preparation step because an incorrect reference filter would contaminate every later result.

### 2.1 Raw and reference-discharge audit

The first audit intentionally starts broad so that the structure of comments can be inspected. A substring search for `"reference discharge"` is useful for diagnosis, but it is **not** used as the final target rule because it also matches non-measurement rows such as post-reference rests.

The production definition of a true capacity reference is therefore:

```text
type == "D"
AND cleaned comment == "reference discharge"
```

The surrounding step sequence is inspected to verify that repeated reference-discharge measurements are genuine parts of the NASA reference procedure rather than accidental duplicates.


In [4]:
print("=== RAW DATA AUDIT ===")

print("Shape:", steps.shape)
print("Number of cells:", steps['cell'].nunique())

print("\nColumns:")
print(steps.columns.tolist())

print("\nStep types:")
print(steps['type'].value_counts())

print("\nReference discharge comments:")
ref_check = steps[
    steps['comment']
        .astype(str)
        .str.contains(
            'reference discharge',
            case=False,
            na=False
        )
]

print("Number of reference-discharge rows:", len(ref_check))
print(
    "Cells with reference discharge:",
    ref_check['cell'].nunique()
)

print("\nExample reference rows:")
display(
    ref_check[
        ['cell', 'step', 'type', 'comment', 'ah']
    ].head(20)
)

=== RAW DATA AUDIT ===
Shape: (1117894, 11)
Number of cells: 28

Columns:
['cell', 'step', 'type', 'comment', 'n', 'dur_s', 'v_start', 'v_end', 'i_mean', 't_max', 'ah']

Step types:
type
R    561172
D    423810
C    132912
Name: count, dtype: int64

Reference discharge comments:
Number of reference-discharge rows: 1943
Cells with reference discharge: 28

Example reference rows:


,cell,step,type,comment,ah
1,RW10,1,D,reference discharge,2.095040
3,RW10,3,D,reference discharge,2.092449
3031,RW10,3031,D,reference discharge,2.051825
3033,RW10,3033,D,reference discharge,2.045415
6057,RW10,6057,D,reference discharge,1.978450
6059,RW10,6059,D,reference discharge,1.980390
9087,RW10,9087,D,reference discharge,1.912779
9089,RW10,9089,D,reference discharge,1.904963
12111,RW10,12111,D,reference discharge,1.866753
12113,RW10,12113,D,reference discharge,1.866285


In [5]:
# ==========================================
# REFERENCE DISCHARGE STRUCTURE AUDIT
# ==========================================

ref_check = (
    steps[
        steps['comment']
        .astype(str)
        .str.contains(
            'reference discharge',
            case=False,
            na=False
        )
    ]
    .copy()
    .sort_values(['cell', 'step'])
)

# Khoảng cách giữa 2 reference-discharge rows liên tiếp
ref_check['gap_from_previous'] = (
    ref_check
    .groupby('cell')['step']
    .diff()
)

print("=== REFERENCE COUNTS PER CELL ===")
print(
    ref_check
    .groupby('cell')
    .size()
    .sort_index()
)

print("\n=== GAP BETWEEN REFERENCE ROWS ===")
print(
    ref_check['gap_from_previous']
    .value_counts()
    .head(20)
)

print("\n=== RW10 FIRST 30 REFERENCE ROWS ===")
display(
    ref_check.loc[
        ref_check['cell'] == 'RW10',
        [
            'cell',
            'step',
            'type',
            'comment',
            'dur_s',
            'v_start',
            'v_end',
            'i_mean',
            't_max',
            'ah',
            'gap_from_previous'
        ]
    ].head(30)
)

=== REFERENCE COUNTS PER CELL ===
cell
RW1      72
RW10    127
RW11    128
RW12    124
RW13     69
RW14     72
RW15     72
RW16     57
RW17     87
RW18     85
RW19     81
RW2      80
RW20     87
RW21     33
RW22     30
RW23     33
RW24     33
RW25     39
RW26     42
RW27     36
RW28     36
RW3      70
RW4      60
RW5      66
RW6      44
RW7      74
RW8      72
RW9     134
dtype: int64

=== GAP BETWEEN REFERENCE ROWS ===
gap_from_previous
1.0       992
3.0       198
2.0       127
3020.0     27
3022.0     26
3016.0     22
3018.0     19
3024.0     16
3014.0     11
3028.0      9
3026.0      7
375.0       4
3012.0      4
674.0       3
828.0       3
3010.0      3
381.0       3
321.0       3
414.0       2
648.0       2
Name: count, dtype: int64

=== RW10 FIRST 30 REFERENCE ROWS ===


,cell,step,type,comment,dur_s,v_start,v_end,i_mean,t_max,ah,gap_from_previous
1,RW10,1,D,reference discharge,7542.95,4.116,3.200,0.999902,26.36092,2.095040,NaN
3,RW10,3,D,reference discharge,7533.92,4.116,3.200,0.999858,26.39203,2.092449,2.0
3031,RW10,3031,D,reference discharge,7387.11,4.112,3.200,0.999935,26.31426,2.051825,3028.0
3033,RW10,3033,D,reference discharge,7363.93,4.111,3.200,0.999949,25.48992,2.045415,2.0
6057,RW10,6057,D,reference discharge,7123.15,4.106,3.200,0.999906,25.72323,1.978450,3024.0
6059,RW10,6059,D,reference discharge,7130.09,4.106,3.200,0.999910,26.46980,1.980390,2.0
9087,RW10,9087,D,reference discharge,6886.67,4.106,3.200,0.999910,25.97208,1.912779,3028.0
9089,RW10,9089,D,reference discharge,6858.58,4.104,3.200,0.999904,25.42771,1.904963,2.0
12111,RW10,12111,D,reference discharge,6720.89,4.103,3.200,0.999923,26.03430,1.866753,3022.0
12113,RW10,12113,D,reference discharge,6719.28,4.104,3.200,0.999912,26.96751,1.866285,2.0


In [6]:
# ==========================================
# TRUE REFERENCE DISCHARGE AUDIT
# ==========================================

comment_clean = (
    steps['comment']
    .astype(str)
    .str.strip()
    .str.lower()
)

true_ref = (
    steps[
        (steps['type'] == 'D') &
        (comment_clean == 'reference discharge')
    ]
    .copy()
    .sort_values(['cell', 'step'])
)

true_ref['gap_from_previous'] = (
    true_ref
    .groupby('cell')['step']
    .diff()
)

print("=== TRUE REFERENCE DISCHARGES ===")
print("Rows:", len(true_ref))
print("Cells:", true_ref['cell'].nunique())

print("\n=== COUNTS PER CELL ===")
print(
    true_ref
    .groupby('cell')
    .size()
    .sort_index()
)

print("\n=== GAPS BETWEEN TRUE REFERENCES ===")
print(
    true_ref['gap_from_previous']
    .value_counts()
    .head(20)
)

print("\n=== CAPACITY CHECK ===")
print(true_ref['ah'].describe())

print("\nZero/negative capacity:")
print((true_ref['ah'] <= 0).sum())

print("\n=== RW10 TRUE REFERENCES ===")
display(
    true_ref.loc[
        true_ref['cell'] == 'RW10',
        [
            'cell',
            'step',
            'type',
            'comment',
            'dur_s',
            'v_start',
            'v_end',
            'i_mean',
            't_max',
            'ah',
            'gap_from_previous'
        ]
    ].head(30)
)

=== TRUE REFERENCE DISCHARGES ===
Rows: 947
Cells: 28

=== COUNTS PER CELL ===
cell
RW1     48
RW10    77
RW11    77
RW12    76
RW13    23
RW14    24
RW15    24
RW16    19
RW17    28
RW18    27
RW19    26
RW2     52
RW20    29
RW21    11
RW22    10
RW23    11
RW24    11
RW25    13
RW26    14
RW27    12
RW28    12
RW3     44
RW4     38
RW5     42
RW6     22
RW7     49
RW8     48
RW9     80
dtype: int64

=== GAPS BETWEEN TRUE REFERENCES ===
gap_from_previous
4.0       198
2.0       127
3021.0     20
3017.0     19
3023.0     17
3019.0     15
3015.0     11
3024.0     11
3022.0     10
3028.0      9
3020.0      7
3025.0      5
3026.0      5
3013.0      4
377.0       4
383.0       4
3018.0      4
675.0       3
3016.0      3
401.0       3
Name: count, dtype: int64

=== CAPACITY CHECK ===
count    947.000000
mean       1.506227
std        0.322156
min        0.692960
25%        1.255316
50%        1.493932
75%        1.755272
max        2.147169
Name: ah, dtype: float64

Zero/negative capacity:

,cell,step,type,comment,dur_s,v_start,v_end,i_mean,t_max,ah,gap_from_previous
1,RW10,1,D,reference discharge,7542.95,4.116,3.2,0.999902,26.36092,2.095040,NaN
3,RW10,3,D,reference discharge,7533.92,4.116,3.2,0.999858,26.39203,2.092449,2.0
3031,RW10,3031,D,reference discharge,7387.11,4.112,3.2,0.999935,26.31426,2.051825,3028.0
3033,RW10,3033,D,reference discharge,7363.93,4.111,3.2,0.999949,25.48992,2.045415,2.0
6057,RW10,6057,D,reference discharge,7123.15,4.106,3.2,0.999906,25.72323,1.978450,3024.0
6059,RW10,6059,D,reference discharge,7130.09,4.106,3.2,0.999910,26.46980,1.980390,2.0
9087,RW10,9087,D,reference discharge,6886.67,4.106,3.2,0.999910,25.97208,1.912779,3028.0
9089,RW10,9089,D,reference discharge,6858.58,4.104,3.2,0.999904,25.42771,1.904963,2.0
12111,RW10,12111,D,reference discharge,6720.89,4.103,3.2,0.999923,26.03430,1.866753,3022.0
12113,RW10,12113,D,reference discharge,6719.28,4.104,3.2,0.999912,26.96751,1.866285,2.0


In [7]:
# ==========================================
# INSPECT FULL STEP SEQUENCE AROUND PAIRS
# ==========================================

cols_show = [
    'cell',
    'step',
    'type',
    'comment',
    'dur_s',
    'v_start',
    'v_end',
    'i_mean',
    't_max',
    'ah'
]

print("=== RW10 STEPS 0-6 ===")
display(
    steps.loc[
        (steps['cell'] == 'RW10') &
        (steps['step'].between(0, 6)),
        cols_show
    ]
)

print("\n=== RW10 STEPS 3029-3035 ===")
display(
    steps.loc[
        (steps['cell'] == 'RW10') &
        (steps['step'].between(3029, 3035)),
        cols_show
    ]
)

print("\n=== RW10 STEPS 6055-6061 ===")
display(
    steps.loc[
        (steps['cell'] == 'RW10') &
        (steps['step'].between(6055, 6061)),
        cols_show
    ]
)

=== RW10 STEPS 0-6 ===


,cell,step,type,comment,dur_s,v_start,v_end,i_mean,t_max,ah
0,RW10,0,C,reference charge,1627.08,4.196,4.200,-0.018988,23.06356,0.008627
1,RW10,1,D,reference discharge,7542.95,4.116,3.200,0.999902,26.36092,2.095040
2,RW10,2,C,reference charge,10495.81,3.494,4.200,-0.719206,26.82753,2.096870
3,RW10,3,D,reference discharge,7533.92,4.116,3.200,0.999858,26.39203,2.092449
4,RW10,4,R,pulsed load (rest),1200.00,4.199,4.197,0.000000,23.15689,0.000000
5,RW10,5,D,pulsed load (discharge),600.00,4.114,3.985,0.999877,23.57683,0.166635
6,RW10,6,R,pulsed load (rest),1200.00,4.061,4.118,0.000000,23.59239,0.000000



=== RW10 STEPS 3029-3035 ===


,cell,step,type,comment,dur_s,v_start,v_end,i_mean,t_max,ah
3029,RW10,3029,C,charge (random walk),300.00,4.023,3.991,-1.499917,29.30055,0.124976
3030,RW10,3030,C,reference charge,10574.58,4.028,4.200,-0.480141,27.52744,1.409602
3031,RW10,3031,D,reference discharge,7387.11,4.112,3.200,0.999935,26.31426,2.051825
3032,RW10,3032,C,reference charge,11887.77,3.519,4.200,-0.622533,26.46980,2.055014
3033,RW10,3033,D,reference discharge,7363.93,4.111,3.200,0.999949,25.48992,2.045415
3034,RW10,3034,R,pulsed charge (rest),1200.00,3.201,3.244,0.000000,22.83026,0.000000
3035,RW10,3035,C,pulsed charge (charge),600.00,3.345,3.838,-0.999890,21.46154,0.166637



=== RW10 STEPS 6055-6061 ===


,cell,step,type,comment,dur_s,v_start,v_end,i_mean,t_max,ah
6055,RW10,6055,D,discharge (random walk),300.00,3.435,3.342,1.499937,31.94104,0.124978
6056,RW10,6056,C,reference charge,13233.24,3.653,4.200,-0.547435,31.21717,2.012052
6057,RW10,6057,D,reference discharge,7123.15,4.106,3.200,0.999906,25.72323,1.978450
6058,RW10,6058,C,reference charge,13021.03,3.524,4.200,-0.548004,25.84765,1.982170
6059,RW10,6059,D,reference discharge,7130.09,4.106,3.200,0.999910,26.46980,1.980390
6060,RW10,6060,R,pulsed load (rest),1200.00,4.199,4.196,0.000000,23.35908,0.000000
6061,RW10,6061,D,pulsed load (discharge),600.00,4.106,3.957,0.999875,24.33896,0.166635


### 2.2 Operational exposure and next-reference SOH

Only operational charge/discharge rows are used as exposure steps. Reference procedures themselves are excluded from cumulative operational throughput.

For each operational row:

1. cumulative Ah throughput is accumulated within the cell;
2. the **nearest future true reference discharge** is identified;
3. its measured capacity becomes the target capacity for that operational row;
4. the cell's maximum reference capacity is used as the initial-capacity denominator;
5. SOH is computed and audited.

Rows without a future reference measurement cannot receive this target and are excluded from the final operational table.

After this construction, the operational table `rw` contains **549,382 rows across 28 cells**.


In [8]:
# ============================================================
# STEP 2 - BUILD USAGE TABLE AND NEXT-REFERENCE SOH TARGET
# ============================================================

import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. Normalise comments
# ------------------------------------------------------------
steps = steps.copy()

steps['comment_clean'] = (
    steps['comment']
    .astype(str)
    .str.strip()
    .str.lower()
)

# ------------------------------------------------------------
# 2. TRUE reference discharge measurements only
# ------------------------------------------------------------
ref = (
    steps[
        (steps['type'] == 'D') &
        (steps['comment_clean'] == 'reference discharge')
    ]
    .copy()
    .sort_values(['cell', 'step'])
)

ref['capacity_ah'] = ref['ah']

print("True reference discharges:", len(ref))
print("Cells with reference:", ref['cell'].nunique())

# Safety checks
assert ref['cell'].nunique() == 28
assert (ref['capacity_ah'] > 0).all()

# ------------------------------------------------------------
# 3. Operational C/D usage steps
#
# Exclude reference procedures themselves.
# ------------------------------------------------------------
usage = (
    steps[
        steps['type'].isin(['C', 'D']) &
        ~steps['comment_clean'].str.contains(
            'reference',
            case=False,
            na=False
        )
    ]
    .copy()
)

print("Operational C/D usage rows:", len(usage))

# ------------------------------------------------------------
# 4. Cumulative OPERATIONAL throughput
#
# Important:
# reference-test Ah is not counted as operational usage.
# ------------------------------------------------------------
usage = usage.sort_values(['cell', 'step'])

usage['cum_ah'] = (
    usage.groupby('cell')['ah']
         .cumsum()
)

# ------------------------------------------------------------
# 5. Attach the NEXT reference discharge to every usage step
# ------------------------------------------------------------
rw = usage.merge(
    ref[
        ['cell', 'step', 'capacity_ah']
    ].rename(
        columns={'step': 'ref_step'}
    ),
    on='cell',
    how='left'
)

# Reference measurement must occur AFTER the usage step
rw = rw[
    rw['ref_step'] > rw['step']
].copy()

# Sort so that the nearest future reference is first
rw = (
    rw.sort_values(
        ['cell', 'step', 'ref_step']
    )
    .drop_duplicates(
        ['cell', 'step'],
        keep='first'
    )
)

# ------------------------------------------------------------
# 6. SOH
# ------------------------------------------------------------
initial_capacity = (
    ref.sort_values(["cell", "step"])
       .groupby("cell")["capacity_ah"]
       .first()
)

rw['capacity_initial_ah'] = (
    rw['cell'].map(initial_capacity)
)

rw['soh'] = (
    rw['capacity_ah'] /
    rw['capacity_initial_ah']
)

# ------------------------------------------------------------
# 7. Final order and save
# ------------------------------------------------------------
rw = rw.sort_values(
    ['cell', 'step']
).reset_index(drop=True)

rw.to_parquet(
    'data/processed/rw_battery.parquet',
    index=False
)

print("\nSaved data/processed/rw_battery.parquet")
print("Shape:", rw.shape)
print("Cells:", rw['cell'].nunique())

True reference discharges: 947
Cells with reference: 28
Operational C/D usage rows: 554522

Saved data/processed/rw_battery.parquet
Shape: (549382, 17)
Cells: 28


In [9]:
first_ref = (
    ref.sort_values(["cell", "step"])
       .groupby("cell", as_index=False)
       .first()[["cell", "step", "capacity_ah"]]
       .rename(columns={
           "step": "first_ref_step",
           "capacity_ah": "first_capacity"
       })
)

max_ref_idx = ref.groupby("cell")["capacity_ah"].idxmax()

max_ref = (
    ref.loc[max_ref_idx, ["cell", "step", "capacity_ah"]]
       .rename(columns={
           "step": "max_ref_step",
           "capacity_ah": "max_capacity"
       })
)

audit = first_ref.merge(max_ref, on="cell")

audit["max_is_first"] = (
    audit["first_ref_step"] == audit["max_ref_step"]
)

print(
    "SOH range:",
    rw["soh"].min(),
    "to",
    rw["soh"].max()
)

print(
    "Rows with SOH > 1:",
    int((rw["soh"] > 1).sum())
)

print(
    "Cells whose maximum capacity occurs after first reference:",
    int((~audit["max_is_first"]).sum()),
    "/",
    len(audit)
)

display(
    audit.loc[
        ~audit["max_is_first"]
    ]
)

SOH range: 0.3309282894664863 to 1.0026357949563953
Rows with SOH > 1: 36
Cells whose maximum capacity occurs after first reference: 3 / 28


,cell,first_ref_step,first_capacity,max_ref_step,max_capacity,max_is_first
8,RW17,2,2.131263,47,2.136881,False
11,RW2,3,1.986917,5,1.988407,False
12,RW20,2,2.129213,47,2.133878,False


In [10]:
# ============================================================
# SOH TARGET AUDIT
# ============================================================

print("=== RW TABLE ===")
print("Shape:", rw.shape)
print("Cells:", rw['cell'].nunique())

print("\n=== SOH SUMMARY ===")
print(rw['soh'].describe())

print("\nMissing SOH:")
print(rw['soh'].isna().sum())

print("\nSOH <= 0:")
print((rw['soh'] <= 0).sum())

print("\nSOH > 1.05:")
print((rw['soh'] > 1.05).sum())

print("\nDuplicate cell-step:")
print(rw.duplicated(['cell', 'step']).sum())

print("\n=== ROWS PER CELL ===")
print(
    rw.groupby('cell')
      .size()
      .sort_index()
)

print("\n=== TARGET REFERENCE CHECK ===")
print(
    "Any ref_step <= step:",
    (rw['ref_step'] <= rw['step']).sum()
)

print("\n=== EXAMPLE RW10 ===")

display(
    rw.loc[
        rw['cell'] == 'RW10',
        [
            'cell',
            'step',
            'comment',
            'ah',
            'cum_ah',
            'ref_step',
            'capacity_ah',
            'capacity_initial_ah',
            'soh'
        ]
    ].iloc[::500].head(20)
)

=== RW TABLE ===
Shape: (549382, 17)
Cells: 28

=== SOH SUMMARY ===
count    549382.000000
mean          0.751134
std           0.157726
min           0.330928
25%           0.617799
50%           0.785382
75%           0.884191
max           1.002636
Name: soh, dtype: float64

Missing SOH:
0

SOH <= 0:
0

SOH > 1.05:
0

Duplicate cell-step:
0

=== ROWS PER CELL ===
cell
RW1      7709
RW10    55267
RW11    54552
RW12    54463
RW13    27584
RW14    22819
RW15    23739
RW16    21117
RW17    15145
RW18    15701
RW19    16637
RW2      7810
RW20    14265
RW21    20957
RW22    18871
RW23    20799
RW24    19605
RW25    10208
RW26     8272
RW27    11077
RW28     9820
RW3      6468
RW4      5440
RW5      5766
RW6      3355
RW7      7970
RW8      7326
RW9     56640
dtype: int64

=== TARGET REFERENCE CHECK ===
Any ref_step <= step: 0

=== EXAMPLE RW10 ===


,cell,step,comment,ah,cum_ah,ref_step,capacity_ah,capacity_initial_ah,soh
7709,RW10,5,pulsed load (discharge),0.166635,0.166635,3031,2.051825,2.09504,0.979373
8209,RW10,1005,charge (random walk),0.062495,76.152166,3031,2.051825,2.09504,0.979373
8709,RW10,2005,charge (random walk),0.133333,146.558364,3031,2.051825,2.09504,0.979373
9209,RW10,3005,discharge (random walk),0.154113,216.852575,3031,2.051825,2.09504,0.979373
9709,RW10,4009,charge (random walk),0.062494,283.291781,6057,1.978450,2.09504,0.944349
10209,RW10,5009,discharge (random walk),0.187463,346.778039,6057,1.978450,2.09504,0.944349
10709,RW10,6009,charge (random walk),0.000000,411.570020,6057,1.978450,2.09504,0.944349
11209,RW10,7013,discharge (random walk),0.062493,473.882716,9087,1.912779,2.09504,0.913004
11709,RW10,8013,discharge (random walk),0.062496,536.728861,9087,1.912779,2.09504,0.913004
12209,RW10,9013,discharge (random walk),0.000000,595.834666,9087,1.912779,2.09504,0.913004


### 2.3 Cleaning log and pre-temperature data-quality evidence

The following cells create the evidence needed to show that cleaning decisions were explicit rather than hidden.

Checks include:

- row counts at major preprocessing stages;
- numeric and categorical summaries;
- missingness;
- physical/range sanity checks;
- duplicate `(cell, step)` keys;
- low-SOH examples;
- per-cell SOH ranges;
- confirmation that low SOH occurs at late accumulated throughput rather than appearing as an obvious random corruption.

Low SOH values are **not removed simply because they are below 0.8 or 0.5**. They are essential for studying end-of-life behavior.


In [11]:
# ============================================================
# STEP 2B - CLEANING / PREPROCESSING LOG
# ============================================================

import pandas as pd
from pathlib import Path

Path('report').mkdir(parents=True, exist_ok=True)

raw_rows = len(steps)
true_reference_rows = len(ref)
usage_candidate_rows = len(usage)
final_rows = len(rw)

rows_without_future_reference = (
    usage_candidate_rows - final_rows
)

duplicate_rows = int(
    rw.duplicated(['cell', 'step']).sum()
)

missing_target_rows = int(
    rw['soh'].isna().sum()
)

cleaning_log = pd.DataFrame([
    {
        'stage': 'Raw step table',
        'rows_before': raw_rows,
        'rows_after': raw_rows,
        'rows_removed': 0,
        'reason': 'Loaded all NASA battery step records'
    },
    {
        'stage': 'Identify true reference discharges',
        'rows_before': raw_rows,
        'rows_after': true_reference_rows,
        'rows_removed': raw_rows - true_reference_rows,
        'reason':
            'Exact comment = reference discharge and type = D; '
            'used only as capacity measurements'
    },
    {
        'stage': 'Select operational charge/discharge steps',
        'rows_before': raw_rows,
        'rows_after': usage_candidate_rows,
        'rows_removed': raw_rows - usage_candidate_rows,
        'reason':
            'Keep C/D operational steps and exclude all reference procedures'
    },
    {
        'stage': 'Require a future reference discharge',
        'rows_before': usage_candidate_rows,
        'rows_after': final_rows,
        'rows_removed': rows_without_future_reference,
        'reason':
            'Rows after the final reference measurement have no future SOH target'
    },
    {
        'stage': 'Check duplicate cell-step keys',
        'rows_before': final_rows,
        'rows_after': final_rows - duplicate_rows,
        'rows_removed': duplicate_rows,
        'reason': 'cell + step must uniquely identify an operational record'
    },
    {
        'stage': 'Check missing SOH target',
        'rows_before': final_rows,
        'rows_after': final_rows - missing_target_rows,
        'rows_removed': missing_target_rows,
        'reason': 'Supervised-learning target must be observed'
    }
])

cleaning_log.to_csv(
    'report/table_cleaning_log.csv',
    index=False
)

display(cleaning_log)

print(
    "\nSaved report/table_cleaning_log.csv"
)

,stage,rows_before,rows_after,rows_removed,reason
0,Raw step table,1117894,1117894,0,Loaded all NASA battery step records
1,Identify true reference discharges,1117894,947,1116947,Exact comment = reference discharge and type =...
2,Select operational charge/discharge steps,1117894,554522,563372,Keep C/D operational steps and exclude all ref...
3,Require a future reference discharge,554522,549382,5140,Rows after the final reference measurement hav...
4,Check duplicate cell-step keys,549382,549382,0,cell + step must uniquely identify an operatio...
5,Check missing SOH target,549382,549382,0,Supervised-learning target must be observed



Saved report/table_cleaning_log.csv


In [12]:
# ============================================================
# STEP 2C - BASIC EDA TABLES
# ============================================================

import numpy as np
import pandas as pd

eda = rw.copy()

# ----------------------------
# Numeric summary
# ----------------------------
numeric_cols = eda.select_dtypes(
    include=np.number
).columns.tolist()

describe_numeric = (
    eda[numeric_cols]
    .describe()
    .T
)

describe_numeric['missing_n'] = (
    eda[numeric_cols]
    .isna()
    .sum()
)

describe_numeric['missing_pct'] = (
    eda[numeric_cols]
    .isna()
    .mean()
    * 100
)

describe_numeric.to_csv(
    'report/table_describe_numeric.csv'
)

# ----------------------------
# Categorical summary
# ----------------------------
categorical_cols = [
    c for c in eda.columns
    if c not in numeric_cols
]

categorical_rows = []

for c in categorical_cols:
    categorical_rows.append({
        'column': c,
        'n_unique': eda[c].nunique(dropna=True),
        'missing_n': eda[c].isna().sum(),
        'missing_pct': eda[c].isna().mean() * 100,
        'top_value':
            eda[c].value_counts(dropna=True).index[0]
            if eda[c].notna().any()
            else None,
        'top_count':
            eda[c].value_counts(dropna=True).iloc[0]
            if eda[c].notna().any()
            else 0
    })

describe_categorical = pd.DataFrame(
    categorical_rows
)

describe_categorical.to_csv(
    'report/table_describe_categorical.csv',
    index=False
)

print("=== NUMERIC SUMMARY ===")
display(describe_numeric)

print("\n=== CATEGORICAL SUMMARY ===")
display(describe_categorical)

=== NUMERIC SUMMARY ===


,count,mean,std,min,25%,50%,75%,max,missing_n,missing_pct
step,549382.0,32533.067043,29222.974702,0.000000,10036.000000,23025.000000,45164.000000,113569.000000,0,0.0
n,549382.0,124.818096,200.836388,1.000000,61.000000,61.000000,203.000000,19129.000000,0,0.0
dur_s,549382.0,425.014720,2158.734776,0.000000,60.000000,60.000000,300.000000,191271.640000,0,0.0
v_start,549382.0,3.669250,0.289879,2.588000,3.461000,3.634000,3.860000,4.814000,0,0.0
v_end,549382.0,3.636651,0.361110,2.588000,3.329000,3.578000,3.917000,4.814000,0,0.0
i_mean,549382.0,1.374357,2.394222,-4.621000,0.499885,2.000033,3.000026,5.128000,0,0.0
t_max,549382.0,-64.515811,627.701242,-4099.447750,26.499652,31.910240,39.708910,58.737730,0,0.0
ah,549382.0,0.088559,0.189794,0.000000,0.024979,0.046477,0.074948,2.144940,0,0.0
cum_ah,549382.0,1326.148596,1008.866390,0.166635,528.690455,1056.430043,1971.975807,3650.196580,0,0.0
ref_step,549382.0,33971.512916,29198.088164,2.000000,11167.000000,24217.000000,46283.000000,113572.000000,0,0.0



=== CATEGORICAL SUMMARY ===


,column,n_unique,missing_n,missing_pct,top_value,top_count
0,cell,28,0,0.0,RW9,56640
1,type,2,0,0.0,D,418109
2,comment,7,0,0.0,discharge (random walk),413234
3,comment_clean,7,0,0.0,discharge (random walk),413234


In [13]:
# ============================================================
# STEP 2D - MISSING VALUE AUDIT
# ============================================================

missing_table = pd.DataFrame({
    'column': eda.columns,
    'missing_n': [
        eda[c].isna().sum()
        for c in eda.columns
    ],
    'missing_pct': [
        eda[c].isna().mean() * 100
        for c in eda.columns
    ]
})

missing_table = (
    missing_table
    .sort_values(
        'missing_pct',
        ascending=False
    )
    .reset_index(drop=True)
)

missing_table['decision'] = 'keep'

# Target can never be missing
missing_table.loc[
    missing_table['column'] == 'soh',
    'decision'
] = 'target: must be complete'

missing_table.to_csv(
    'report/table_missing.csv',
    index=False
)

display(missing_table)

,column,missing_n,missing_pct,decision
0,cell,0,0.0,keep
1,step,0,0.0,keep
2,type,0,0.0,keep
3,comment,0,0.0,keep
4,n,0,0.0,keep
5,dur_s,0,0.0,keep
6,v_start,0,0.0,keep
7,v_end,0,0.0,keep
8,i_mean,0,0.0,keep
9,t_max,0,0.0,keep


In [14]:
# ============================================================
# STEP 2E - PHYSICAL / RANGE AUDIT
# ============================================================

range_audit = pd.DataFrame([
    {
        'check': 'SOH <= 0',
        'count': int((eda['soh'] <= 0).sum())
    },
    {
        'check': 'SOH > 1',
        'count': int((eda['soh'] > 1).sum())
    },
    {
        'check': 'Negative duration',
        'count': int((eda['dur_s'] < 0).sum())
    },
    {
        'check': 'Negative Ah',
        'count': int((eda['ah'] < 0).sum())
    },
    {
        'check': 'Missing cell',
        'count': int(eda['cell'].isna().sum())
    },
    {
        'check': 'Duplicate cell-step',
        'count': int(
            eda.duplicated(['cell', 'step']).sum()
        )
    },
    {
        'check': 'Reference occurs before/equal usage step',
        'count': int(
            (eda['ref_step'] <= eda['step']).sum()
        )
    }
])

range_audit.to_csv(
    'report/table_quality_checks.csv',
    index=False
)

display(range_audit)

,check,count
0,SOH <= 0,0
1,SOH > 1,36
2,Negative duration,0
3,Negative Ah,0
4,Missing cell,0
5,Duplicate cell-step,0
6,Reference occurs before/equal usage step,0


In [15]:
# ============================================================
# LOW-SOH AUDIT
# ============================================================

low_soh = (
    eda[
        [
            'cell',
            'step',
            'ref_step',
            'cum_ah',
            'capacity_ah',
            'capacity_initial_ah',
            'soh'
        ]
    ]
    .sort_values('soh')
    .head(30)
)

display(low_soh)

,cell,step,ref_step,cum_ah,capacity_ah,capacity_initial_ah,soh
113049,RW11,100410,103347,3511.986801,0.69296,2.093988,0.330928
113018,RW11,100348,103347,3510.070970,0.69296,2.093988,0.330928
113017,RW11,100346,103347,3510.067695,0.69296,2.093988,0.330928
113016,RW11,100343,103347,3509.997813,0.69296,2.093988,0.330928
113015,RW11,100341,103347,3509.895801,0.69296,2.093988,0.330928
113014,RW11,100339,103347,3509.729164,0.69296,2.093988,0.330928
113013,RW11,100337,103347,3509.562528,0.69296,2.093988,0.330928
113012,RW11,100335,103347,3509.395888,0.69296,2.093988,0.330928
113011,RW11,100333,103347,3509.229246,0.69296,2.093988,0.330928
113010,RW11,100331,103347,3509.062606,0.69296,2.093988,0.330928


In [16]:
# ============================================================
# PER-CELL SOH RANGE AUDIT
# ============================================================

cell_soh_summary = (
    rw.groupby('cell')
      .agg(
          rows=('soh', 'size'),
          soh_min=('soh', 'min'),
          soh_max=('soh', 'max'),
          soh_mean=('soh', 'mean'),
          cum_ah_max=('cum_ah', 'max'),
          step_max=('step', 'max')
      )
      .sort_values('soh_min')
      .reset_index()
)

display(cell_soh_summary)

cell_soh_summary.to_csv(
    'report/table_soh_by_cell.csv',
    index=False
)

,cell,rows,soh_min,soh_max,soh_mean,cum_ah_max,step_max
0,RW11,54552,0.330928,0.981573,0.641218,3650.196580,109384
1,RW9,56640,0.356938,0.974653,0.607980,3519.722557,113569
2,RW10,55267,0.401921,0.979373,0.636228,3568.479333,110814
3,RW14,22819,0.415794,1.000000,0.746604,1449.885777,45798
4,RW15,23739,0.442830,1.000000,0.744557,1506.786810,47638
5,RW16,21117,0.449564,1.000000,0.760327,1364.021676,42359
6,RW20,14265,0.488782,1.002191,0.785280,1428.772773,28729
7,RW19,16637,0.504186,1.000000,0.794022,1581.481886,33689
8,RW13,27584,0.513416,1.000000,0.785668,1780.184134,55321
9,RW17,15145,0.514201,1.002636,0.795608,1398.655794,30711


In [17]:
# ============================================================
# WHERE DOES MINIMUM SOH OCCUR?
# ============================================================

min_soh_rows = (
    rw.loc[
        rw.groupby('cell')['soh'].idxmin(),
        [
            'cell',
            'step',
            'cum_ah',
            'ref_step',
            'capacity_ah',
            'capacity_initial_ah',
            'soh'
        ]
    ]
    .sort_values('soh')
    .reset_index(drop=True)
)

display(min_soh_rows)

,cell,step,cum_ah,ref_step,capacity_ah,capacity_initial_ah,soh
0,RW11,100331,3509.062606,103347,0.692960,2.093988,0.330928
1,RW9,110558,3483.629935,113572,0.748762,2.097739,0.356938
2,RW10,107803,3525.517174,110817,0.842042,2.095040,0.401921
3,RW14,45150,1433.960016,45800,0.879564,2.115386,0.415794
4,RW15,46894,1488.085520,47640,0.938390,2.119074,0.442830
5,RW16,41713,1347.672799,42361,0.950218,2.113643,0.449564
6,RW20,28108,1410.417333,28424,1.040720,2.129213,0.488782
7,RW19,33389,1572.063425,33691,1.074933,2.132017,0.504186
8,RW13,54433,1755.580394,55323,1.093057,2.128989,0.513416
9,RW17,30409,1389.149720,30713,1.095899,2.131263,0.514201


## 3. Temperature anomaly audit and cleaning

The raw temperature field contains a small but important sensor-quality problem.

Before cleaning, the operational table includes values around **−4099 °C**, which are physically implausible for these battery experiments and behave like sentinel/sensor-invalid values. The audit found **13,315 negative temperature rows** in `rw`, about **2.42%** of the operational table.

### Cleaning policy

The project does **not** delete the whole operational step. Instead:

- `temp_invalid = 1` marks the anomaly;
- negative `t_max` values are replaced by `NaN`;
- voltage, current, duration, Ah throughput, and target information on that row are retained.

This distinction matters because deleting the full row would throw away valid operational information.

The cleaned temperature values are **not median-imputed in Notebook A**. Notebook B must fit any imputer using training cells only, then apply it to validation/test cells. This prevents cross-cell leakage.

A very small number of non-negative temperatures near 0 °C are retained because the current evidence is insufficient to classify them as invalid.


In [18]:
# ============================================================
# TEMPERATURE DIAGNOSTIC
# ============================================================

print("=== t_max DESCRIBE ===")
print(
    rw['t_max'].describe(
        percentiles=[
            0.01, 0.05, 0.10,
            0.25, 0.50, 0.75,
            0.90, 0.95, 0.99
        ]
    )
)

print("\n=== TEMPERATURE COUNTS ===")
print("t_max < -100 :", (rw['t_max'] < -100).sum())
print("t_max < 0    :", (rw['t_max'] < 0).sum())
print("0 <= t_max < 10 :", ((rw['t_max'] >= 0) & (rw['t_max'] < 10)).sum())
print("10 <= t_max <= 80:",
      ((rw['t_max'] >= 10) & (rw['t_max'] <= 80)).sum())
print("t_max > 80   :", (rw['t_max'] > 80).sum())

print("\n=== MINIMUM TEMPERATURE ROWS ===")
display(
    rw[
        [
            'cell', 'step', 'type', 'comment',
            'dur_s', 'i_mean', 't_max', 'ah', 'soh'
        ]
    ]
    .sort_values('t_max')
    .head(30)
)

print("\n=== TEMPERATURE BY COMMENT ===")
temp_by_comment = (
    rw.groupby('comment')
      .agg(
          rows=('t_max', 'size'),
          temp_mean=('t_max', 'mean'),
          temp_min=('t_max', 'min'),
          temp_max=('t_max', 'max')
      )
      .sort_values('rows', ascending=False)
)

display(temp_by_comment.head(30))

=== t_max DESCRIBE ===
count    549382.000000
mean        -64.515811
std         627.701242
min       -4099.447750
1%        -4094.098140
5%           20.806070
10%          23.019460
25%          26.499652
50%          31.910240
75%          39.708910
90%          45.799610
95%          47.670610
99%          51.549250
max          58.737730
Name: t_max, dtype: float64

=== TEMPERATURE COUNTS ===
t_max < -100 : 13164
t_max < 0    : 13315
0 <= t_max < 10 : 52
10 <= t_max <= 80: 536015
t_max > 80   : 0

=== MINIMUM TEMPERATURE ROWS ===


,cell,step,type,comment,dur_s,i_mean,t_max,ah,soh
284313,RW18,4061,D,discharge (random walk),60.00,2.500082,-4099.44775,0.041639,0.921366
284312,RW18,4058,D,discharge (random walk),23.15,4.500040,-4099.44775,0.028887,0.921366
284311,RW18,4056,D,discharge (random walk),60.00,2.000049,-4099.44775,0.033312,0.921366
284310,RW18,4054,D,discharge (random walk),60.00,3.000000,-4099.44775,0.049966,0.921366
284309,RW18,4052,D,discharge (random walk),60.00,2.500016,-4099.44775,0.041639,0.921366
284334,RW18,4104,D,discharge (random walk),60.00,4.499918,-4099.44775,0.074948,0.921366
284333,RW18,4102,D,discharge (random walk),60.00,3.999951,-4099.44775,0.066621,0.921366
284332,RW18,4100,D,discharge (random walk),60.00,3.500049,-4099.44775,0.058295,0.921366
284331,RW18,4098,D,discharge (random walk),60.00,4.499984,-4099.44775,0.074949,0.921366
284330,RW18,4096,D,discharge (random walk),60.00,0.999984,-4099.44775,0.016655,0.921366



=== TEMPERATURE BY COMMENT ===


,rows,temp_mean,temp_min,temp_max
comment,,,,
discharge (random walk),413234,-76.108313,-4099.44775,58.73773
charge (random walk),109647,30.409477,18.14507,46.59140
charge (after random walk discharge),20761,-275.861112,-4099.44775,54.34815
pulsed load (discharge),4848,-252.345062,-4099.44775,46.38136
pulsed charge (charge),853,-409.462512,-4093.92700,37.76810
low current discharge at 0.04A,27,-584.316989,-4094.09814,36.61286
low current charge,12,-1001.763267,-4093.92700,37.29060


In [19]:
# ============================================================
# TEMPERATURE SENSOR CLEANING
# ============================================================

rw = rw.copy()

# Mark original invalid temperature readings.
# Negative maximum temperature is not physically plausible
# in the observed operating range of this dataset and includes
# repeated sentinel-like values around -4099.
rw['temp_invalid'] = (
    rw['t_max'] < 0
).astype(int)

# Do NOT delete the row.
# Only mark the invalid temperature measurement as missing.
rw.loc[
    rw['t_max'] < 0,
    't_max'
] = np.nan

print("Invalid temperature rows:",
      rw['temp_invalid'].sum())

print("\nTemperature after cleaning:")
print(rw['t_max'].describe())

print("\nMissing t_max after cleaning:")
print(rw['t_max'].isna().sum())

Invalid temperature rows: 13315

Temperature after cleaning:
count    536067.000000
mean         33.522699
std           8.269774
min           0.159350
25%          26.863840
50%          32.191000
75%          39.986170
max          58.737730
Name: t_max, dtype: float64

Missing t_max after cleaning:
13315


In [20]:
# ============================================================
# UPDATED DATA QUALITY AUDIT
# ============================================================

quality_updated = pd.DataFrame([
    {
        'check': 'Missing SOH',
        'count': int(rw['soh'].isna().sum())
    },
    {
        'check': 'Invalid SOH',
        'count': int(
            ((rw['soh'] <= 0) | (rw['soh'] > 1)).sum()
        )
    },
    {
        'check': 'Duplicate cell-step',
        'count': int(
            rw.duplicated(['cell', 'step']).sum()
        )
    },
    {
        'check': 'Negative duration',
        'count': int((rw['dur_s'] < 0).sum())
    },
    {
        'check': 'Negative Ah',
        'count': int((rw['ah'] < 0).sum())
    },
    {
        'check': 'Temperature sensor invalid',
        'count': int(rw['temp_invalid'].sum())
    },
    {
        'check': 'Missing cleaned temperature',
        'count': int(rw['t_max'].isna().sum())
    }
])

display(quality_updated)

quality_updated.to_csv(
    'report/table_quality_checks.csv',
    index=False
)

,check,count
0,Missing SOH,0
1,Invalid SOH,36
2,Duplicate cell-step,0
3,Negative duration,0
4,Negative Ah,0
5,Temperature sensor invalid,13315
6,Missing cleaned temperature,13315


In [21]:
# ============================================================
# UPDATED MISSING VALUE TABLE AFTER TEMPERATURE CLEANING
# ============================================================

missing_table = pd.DataFrame({
    "column": rw.columns,
    "missing_n": [
        int(rw[c].isna().sum())
        for c in rw.columns
    ],
    "missing_pct": [
        float(rw[c].isna().mean() * 100)
        for c in rw.columns
    ]
})

missing_table = (
    missing_table
    .sort_values(
        "missing_pct",
        ascending=False
    )
    .reset_index(drop=True)
)

missing_table["decision"] = "keep"

missing_table.loc[
    missing_table["column"] == "soh",
    "decision"
] = "target: must be complete"

missing_table.loc[
    missing_table["column"] == "t_max",
    "decision"
] = (
    "keep as missing; impute using training cells only in Notebook_B"
)

missing_table.to_csv(
    "report/table_missing.csv",
    index=False
)

display(missing_table)

,column,missing_n,missing_pct,decision
0,t_max,13315,2.423632,keep as missing; impute using training cells o...
1,cell,0,0.000000,keep
2,step,0,0.000000,keep
3,type,0,0.000000,keep
4,n,0,0.000000,keep
5,comment,0,0.000000,keep
6,dur_s,0,0.000000,keep
7,v_start,0,0.000000,keep
8,v_end,0,0.000000,keep
9,i_mean,0,0.000000,keep


### 3.1 Save the cleaned operational table

After temperature cleaning and quality checks, the operational table is written to:

```text
data/processed/rw_battery.parquet
```

This file is the clean source used by the SQL stage below.


In [22]:
rw.to_parquet(
    'data/processed/rw_battery.parquet',
    index=False
)

print(
    "Re-saved cleaned rw_battery.parquet:",
    rw.shape
)

Re-saved cleaned rw_battery.parquet: (549382, 18)


## 4. SQL analysis and 50-step feature engineering

The cleaned parquet is loaded into DuckDB as table `rb`.

Four SQL tasks are used:

**Q1 — Throughput bins.** Summarize mean/min/max SOH by cell and 100-Ah cumulative-throughput bins.

**Q2 — Health-state description.** Compare descriptive load statistics across `healthy`, `aged`, and `end_of_life` SOH states. These are associations, not causal effects.

**Q3 — Rolling load history.** Create features using the **current operational step plus the preceding 49 operational steps**:

- `dv`: voltage change across the current step;
- `i_ma50`: mean absolute current over 50 steps;
- `t_max50`: maximum valid temperature over 50 steps;
- `ah50`: Ah throughput over 50 steps;
- `temp_valid_n50`: number of non-missing temperature values in the window;
- `history_n`: number of operational rows in the window.

Only rows with `history_n == 50` are retained. This removes the first 49 operational rows of each cell and prevents partial-history rows from being presented as full-history features.

**Q4 — Final SQL quality check.** Verify row count, cell count, SOH validity, history length, duration, and Ah.

The final feature table contains **548,010 rows × 17 columns across 28 cells**.


In [23]:
# ============================================================
# STEP 3A - LOAD CLEAN DATA INTO DUCKDB
# ============================================================

from pathlib import Path
import duckdb

clean_path = Path("data/processed/rw_battery.parquet")
db_path = Path("data/processed/ady.duckdb")

assert clean_path.exists(), \
    f"Cannot find: {clean_path.resolve()}"

con = duckdb.connect(str(db_path))

con.execute(f"""
CREATE OR REPLACE TABLE rb AS
SELECT *
FROM read_parquet('{clean_path.as_posix()}')
""")

check_rb = con.execute("""
SELECT
    COUNT(*) AS rows,
    COUNT(DISTINCT cell) AS cells,
    MIN(soh) AS soh_min,
    MAX(soh) AS soh_max,
    MIN(cum_ah) AS cum_ah_min,
    MAX(cum_ah) AS cum_ah_max
FROM rb
""").df()

display(check_rb)

,rows,cells,soh_min,soh_max,cum_ah_min,cum_ah_max
0,549382,28,0.330928,1.002636,0.166635,3650.19658


In [24]:
# ============================================================
# Q1 - SOH VERSUS CUMULATIVE THROUGHPUT
# ============================================================

q1 = con.execute("""
SELECT
    cell,
    FLOOR(cum_ah / 100.0) * 100.0 AS ah_bin,
    COUNT(*) AS n_steps,
    AVG(soh) AS mean_soh,
    MIN(soh) AS min_soh,
    MAX(soh) AS max_soh
FROM rb
GROUP BY cell, ah_bin
HAVING COUNT(*) >= 20
ORDER BY cell, ah_bin
""").df()

q1.to_csv(
    'report/table_q1.csv',
    index=False
)

display(q1.head(30))

print("Q1 rows:", len(q1))
print("Cells:", q1['cell'].nunique())

,cell,ah_bin,n_steps,mean_soh,min_soh,max_soh
0,RW1,0.0,350,0.979182,0.979063,1.000000
1,RW1,100.0,370,0.963608,0.955433,0.979063
2,RW1,200.0,392,0.944735,0.934774,0.955433
3,RW1,300.0,384,0.925645,0.913136,0.934774
4,RW1,400.0,375,0.908661,0.894068,0.913136
5,RW1,500.0,366,0.892099,0.872871,0.894068
6,RW1,600.0,352,0.872871,0.872871,0.872871
7,RW1,700.0,355,0.859453,0.856940,0.872871
8,RW1,800.0,383,0.846047,0.842352,0.856940
9,RW1,900.0,373,0.828218,0.821997,0.842352


Q1 rows: 499
Cells: 28


In [25]:
# ============================================================
# Q2 - LOAD SEVERITY BY HEALTH STATE
# ============================================================

q2 = con.execute("""
SELECT
    CASE
        WHEN soh > 0.90 THEN 'healthy'
        WHEN soh > 0.80 THEN 'aged'
        ELSE 'end_of_life'
    END AS health_state,

    COUNT(*) AS n_steps,

    COUNT(t_max) AS n_valid_temperature,

    AVG(ABS(i_mean)) AS mean_abs_current_a,
    AVG(t_max) AS mean_max_temperature_c,
    AVG(dur_s) AS mean_duration_s,
    AVG(ah) AS mean_step_ah,
    AVG(cum_ah) AS mean_cumulative_ah

FROM rb

GROUP BY health_state

ORDER BY
    CASE health_state
        WHEN 'healthy' THEN 1
        WHEN 'aged' THEN 2
        WHEN 'end_of_life' THEN 3
    END
""").df()

q2.to_csv(
    'report/table_q2.csv',
    index=False
)

display(q2)

,health_state,n_steps,n_valid_temperature,mean_abs_current_a,mean_max_temperature_c,mean_duration_s,mean_step_ah,mean_cumulative_ah
0,healthy,119080,116853,2.476785,37.513314,349.294400,0.103370,255.077035
1,aged,130969,127201,2.506167,36.164231,447.253392,0.108608,762.061299
2,end_of_life,299333,292013,2.410750,30.775148,445.407389,0.073895,1999.048513


In [26]:
# ============================================================
# Q3 - 50-STEP LOAD-HISTORY FEATURES
# ============================================================

con.execute("""
CREATE OR REPLACE TABLE feat_raw AS

SELECT
    cell,
    step,
    type,

    dur_s,
    v_start,
    v_end,
    i_mean,
    t_max,
    ah,
    cum_ah,

    soh,

    v_start - v_end AS dv,

    AVG(ABS(i_mean)) OVER (
        PARTITION BY cell
        ORDER BY step
        ROWS BETWEEN 49 PRECEDING AND CURRENT ROW
    ) AS i_ma50,

    MAX(t_max) OVER (
        PARTITION BY cell
        ORDER BY step
        ROWS BETWEEN 49 PRECEDING AND CURRENT ROW
    ) AS t_max50,

    SUM(ah) OVER (
        PARTITION BY cell
        ORDER BY step
        ROWS BETWEEN 49 PRECEDING AND CURRENT ROW
    ) AS ah50,

    COUNT(t_max) OVER (
        PARTITION BY cell
        ORDER BY step
        ROWS BETWEEN 49 PRECEDING AND CURRENT ROW
    ) AS temp_valid_n50,

    COUNT(*) OVER (
        PARTITION BY cell
        ORDER BY step
        ROWS BETWEEN 49 PRECEDING AND CURRENT ROW
    ) AS history_n

FROM rb
""")

In [27]:
# ============================================================
# FINAL FEATURE TABLE
# ============================================================

con.execute("""
CREATE OR REPLACE TABLE feat AS

SELECT *
FROM feat_raw

WHERE history_n = 50
""")

feat = con.execute("""
SELECT *
FROM feat
ORDER BY cell, step
""").df()

print("Feature table shape:", feat.shape)
print("Cells:", feat['cell'].nunique())
print("Columns:", len(feat.columns))

display(feat.head())

Feature table shape: (548010, 17)
Cells: 28
Columns: 17


,cell,step,type,dur_s,v_start,v_end,i_mean,t_max,ah,cum_ah,soh,dv,i_ma50,t_max50,ah50,temp_valid_n50,history_n
0,RW1,105,D,187.41,3.321,3.200,2.999942,29.92934,0.156138,13.923569,0.979063,0.121,1.737647,36.67751,13.923569,50,50
1,RW1,108,C,1800.00,3.843,4.200,-1.957801,29.85138,0.979266,14.902835,0.979063,-0.357,1.776005,36.67751,12.807810,50,50
2,RW1,110,D,300.00,3.605,3.389,2.999930,29.88256,0.249961,15.152796,0.979063,0.216,1.835206,36.67751,10.937189,50,50
3,RW1,112,D,300.00,3.315,3.263,3.500033,34.80922,0.291630,15.444426,0.979063,0.052,1.885208,36.67751,11.062181,50,50
4,RW1,114,D,300.00,3.338,3.286,2.999904,35.64300,0.249958,15.694384,0.979063,0.052,1.925208,36.67751,11.145499,50,50


In [28]:
# ============================================================
# Q4 - SQL QUALITY CHECK
# ============================================================

q4 = con.execute("""
SELECT
    COUNT(*) AS rows,
    COUNT(DISTINCT cell) AS cells,

    SUM(
        CASE
            WHEN soh IS NULL THEN 1
            ELSE 0
        END
    ) AS missing_soh,

    SUM(
        CASE
            WHEN soh <= 0 OR soh > 1 THEN 1
            ELSE 0
        END
    ) AS invalid_soh,

    SUM(
        CASE
            WHEN history_n <> 50 THEN 1
            ELSE 0
        END
    ) AS invalid_history,

    SUM(
        CASE
            WHEN dur_s < 0 THEN 1
            ELSE 0
        END
    ) AS negative_duration,

    SUM(
        CASE
            WHEN ah < 0 THEN 1
            ELSE 0
        END
    ) AS negative_ah

FROM feat
""").df()

q4.to_csv(
    'report/table_q4.csv',
    index=False
)

display(q4)

,rows,cells,missing_soh,invalid_soh,invalid_history,negative_duration,negative_ah
0,548010,28,0.0,0.0,0.0,0.0,0.0


In [29]:
# ============================================================
# SAVE SQL QUERIES FOR REPRODUCIBILITY
# ============================================================

from pathlib import Path

Path("sql").mkdir(
    parents=True,
    exist_ok=True
)

sql_text = r"""
-- ============================================================
-- Q1 - SOH versus cumulative operational throughput
-- ============================================================

SELECT
    cell,
    FLOOR(cum_ah / 100) * 100 AS ah_bin,
    COUNT(*) AS n_steps,
    AVG(soh) AS mean_soh,
    MIN(soh) AS min_soh,
    MAX(soh) AS max_soh
FROM rb
GROUP BY cell, ah_bin
HAVING COUNT(*) >= 20
ORDER BY cell, ah_bin;


-- ============================================================
-- Q2 - Load severity by health state
-- ============================================================

SELECT
    CASE
        WHEN soh > 0.9 THEN 'healthy'
        WHEN soh > 0.8 THEN 'aged'
        ELSE 'EOL'
    END AS health_state,

    COUNT(*) AS n_steps,
    COUNT(t_max) AS n_valid_temp,

    AVG(ABS(i_mean)) AS mean_abs_current,
    AVG(t_max) AS mean_max_temp,
    AVG(dur_s) AS mean_duration_s,
    AVG(ah) AS mean_step_ah,
    AVG(cum_ah) AS mean_cum_ah

FROM rb
GROUP BY health_state
ORDER BY
    CASE health_state
        WHEN 'healthy' THEN 1
        WHEN 'aged' THEN 2
        WHEN 'EOL' THEN 3
    END;


-- ============================================================
-- Q3 - 50-step rolling load-history features
-- ============================================================

CREATE OR REPLACE TABLE feat_raw AS

SELECT
    cell,
    step,
    type,

    dur_s,
    v_start,
    v_end,
    i_mean,
    t_max,
    ah,
    cum_ah,

    soh,

    v_start - v_end AS dv,

    AVG(ABS(i_mean)) OVER (
        PARTITION BY cell
        ORDER BY step
        ROWS BETWEEN 49 PRECEDING AND CURRENT ROW
    ) AS i_ma50,

    MAX(t_max) OVER (
        PARTITION BY cell
        ORDER BY step
        ROWS BETWEEN 49 PRECEDING AND CURRENT ROW
    ) AS t_max50,

    SUM(ah) OVER (
        PARTITION BY cell
        ORDER BY step
        ROWS BETWEEN 49 PRECEDING AND CURRENT ROW
    ) AS ah50,

    COUNT(t_max) OVER (
        PARTITION BY cell
        ORDER BY step
        ROWS BETWEEN 49 PRECEDING AND CURRENT ROW
    ) AS temp_valid_n50,

    COUNT(*) OVER (
        PARTITION BY cell
        ORDER BY step
        ROWS BETWEEN 49 PRECEDING AND CURRENT ROW
    ) AS history_n

FROM rb;


CREATE OR REPLACE TABLE feat AS

SELECT *
FROM feat_raw
WHERE history_n = 50;


-- ============================================================
-- Q4 - Final feature-table quality checks
-- ============================================================

SELECT
    COUNT(*) AS rows,
    COUNT(DISTINCT cell) AS cells,

    SUM(
        CASE
            WHEN soh IS NULL THEN 1
            ELSE 0
        END
    ) AS missing_soh,

    SUM(
        CASE
            WHEN soh <= 0 OR soh > 1 THEN 1
            ELSE 0
        END
    ) AS invalid_soh,

    SUM(
        CASE
            WHEN history_n <> 50 THEN 1
            ELSE 0
        END
    ) AS invalid_history,

    SUM(
        CASE
            WHEN dur_s < 0 THEN 1
            ELSE 0
        END
    ) AS negative_duration,

    SUM(
        CASE
            WHEN ah < 0 THEN 1
            ELSE 0
        END
    ) AS negative_ah

FROM feat;
"""

sql_path = Path(
    "sql/queries.sql"
)

sql_path.write_text(
    sql_text,
    encoding="utf-8"
)

print(
    "Saved:",
    sql_path.resolve()
)

Saved: E:\PJ_ADY\sql\queries.sql


## 5. RQ1 statistical analysis

RQ1 is evaluated at the **cell level** so that the cross-battery pattern is visible instead of being hidden inside one pooled correlation.

### RQ1A — SOH versus cumulative throughput

For every cell, Spearman's rank correlation measures the monotonic association between cumulative operational throughput and SOH. A descriptive linear slope is also reported as SOH change per 100 Ah.

Across 28 cells, the median Spearman correlation is approximately **−0.996**, showing a very strong and consistent inverse monotonic relationship.

### RQ1B — Cross-cell consistency

A one-sided Wilcoxon signed-rank test evaluates whether the 28 cell-wise correlations are systematically below zero.

Result: **Wilcoxon statistic = 0; p = 3.73 × 10⁻⁹**. In the report this should be written as **p < .001**, not `p = 0`. Values displayed as `0.0` in some per-cell tables are numerical underflow from extremely small p-values.

### RQ1C — Raw recent-load associations

Median cell-wise raw Spearman correlations with SOH are approximately:

- `ah50`: **+0.431**
- `i_ma50`: **+0.416**
- `t_max50`: **+0.159**

These positive raw correlations must **not** be interpreted as "heavier load improves battery health". They are confounded by life stage, experimental protocol, cumulative exposure, and cell-to-cell differences.

### RQ1D — Partial Spearman controlling cumulative throughput

After rank-residualizing both SOH and each load-history feature against cumulative throughput, the median partial correlations become:

- `ah50`: **−0.066**
- `i_ma50`: **+0.122**
- `t_max50`: **+0.029**

The main RQ1 interpretation is therefore:

> **Accumulated operational throughput dominates the monotonic degradation signal in this dataset. The three 50-step load-history summaries show much weaker residual monotonic associations with SOH once cumulative throughput is controlled.**

This is a descriptive association result, not a causal claim. Notebook B will use ablation experiments to test whether load-history features still add nonlinear predictive value.


In [30]:
# ============================================================
# RQ1A - PER-CELL SOH VS CUMULATIVE THROUGHPUT
# ============================================================

import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from pathlib import Path

Path("report").mkdir(parents=True, exist_ok=True)

rq1a_rows = []

for cell, g in feat.groupby("cell"):

    g = (
        g[["cum_ah", "soh"]]
        .dropna()
        .sort_values("cum_ah")
    )

    rho, p_value = spearmanr(
        g["cum_ah"],
        g["soh"]
    )

    slope, intercept = np.polyfit(
        g["cum_ah"],
        g["soh"],
        1
    )

    rq1a_rows.append({
        "cell": cell,
        "n_rows": len(g),
        "spearman_rho": rho,
        "p_value": p_value,
        "soh_change_per_100ah": slope * 100
    })

rq1a = pd.DataFrame(rq1a_rows)

rq1a = rq1a.sort_values(
    "spearman_rho"
).reset_index(drop=True)

rq1a.to_csv(
    "report/table_rq1a_throughput.csv",
    index=False
)

display(rq1a)

print(
    "\nMedian Spearman rho:",
    rq1a["spearman_rho"].median()
)

print(
    "Mean Spearman rho:",
    rq1a["spearman_rho"].mean()
)

print(
    "Saved:",
    Path(
        "report/table_rq1a_throughput.csv"
    ).resolve()
)

,cell,n_rows,spearman_rho,p_value,soh_change_per_100ah
0,RW12,54414,-0.998935,0.0,-0.012988
1,RW9,56591,-0.998213,0.0,-0.016834
2,RW3,6419,-0.997114,0.0,-0.022839
3,RW15,23690,-0.997017,0.0,-0.035026
4,RW20,14216,-0.996977,0.0,-0.032393
5,RW13,27535,-0.996936,0.0,-0.024186
6,RW14,22770,-0.996756,0.0,-0.038079
7,RW19,16588,-0.996726,0.0,-0.023594
8,RW5,5717,-0.996694,0.0,-0.023152
9,RW26,8223,-0.996423,0.0,-0.016350



Median Spearman rho: -0.9961377425742599
Mean Spearman rho: -0.9944749702609844
Saved: E:\PJ_ADY\report\table_rq1a_throughput.csv


In [31]:
# ============================================================
# RQ1B - CROSS-CELL CONSISTENCY TEST
# ============================================================

import pandas as pd
from scipy.stats import wilcoxon
from pathlib import Path

rho_values = rq1a["spearman_rho"].dropna()

# Test whether cell-wise correlations are consistently below zero
wilcoxon_result = wilcoxon(
    rho_values,
    alternative="less"
)

rq1b = pd.DataFrame([{
    "n_cells": len(rho_values),
    "median_rho": rho_values.median(),
    "mean_rho": rho_values.mean(),
    "min_rho": rho_values.min(),
    "max_rho": rho_values.max(),
    "wilcoxon_statistic": wilcoxon_result.statistic,
    "p_value": wilcoxon_result.pvalue
}])

rq1b.to_csv(
    "report/table_rq1b_throughput_test.csv",
    index=False
)

display(rq1b)

print(
    "Saved:",
    Path(
        "report/table_rq1b_throughput_test.csv"
    ).resolve()
)

,n_cells,median_rho,mean_rho,min_rho,max_rho,wilcoxon_statistic,p_value
0,28,-0.996138,-0.994475,-0.998935,-0.9741,0.0,3.725290e-09


Saved: E:\PJ_ADY\report\table_rq1b_throughput_test.csv


In [32]:
# ============================================================
# RQ1C - RAW LOAD-HISTORY ASSOCIATIONS
# ============================================================

import pandas as pd
from scipy.stats import spearmanr
from pathlib import Path

features_rq1c = [
    "ah50",
    "i_ma50",
    "t_max50"
]

rq1c_rows = []

for cell, g in feat.groupby("cell"):

    for feature in features_rq1c:

        temp = (
            g[[feature, "soh"]]
            .dropna()
        )

        rho, p_value = spearmanr(
            temp[feature],
            temp["soh"]
        )

        rq1c_rows.append({
            "cell": cell,
            "feature": feature,
            "n_rows": len(temp),
            "rho": rho,
            "p_value": p_value
        })

rq1c = pd.DataFrame(rq1c_rows)

rq1c_summary = (
    rq1c
    .groupby("feature")["rho"]
    .agg(
        n_cells="count",
        median_rho="median",
        mean_rho="mean",
        min_rho="min",
        max_rho="max"
    )
    .reset_index()
)

rq1c.to_csv(
    "report/table_rq1c_load_per_cell.csv",
    index=False
)

rq1c_summary.to_csv(
    "report/table_rq1c_load_summary.csv",
    index=False
)

display(rq1c_summary)

print(
    "Saved:",
    Path(
        "report/table_rq1c_load_per_cell.csv"
    ).resolve()
)

print(
    "Saved:",
    Path(
        "report/table_rq1c_load_summary.csv"
    ).resolve()
)

,feature,n_cells,median_rho,mean_rho,min_rho,max_rho
0,ah50,28,0.431202,0.468548,-0.043544,0.912219
1,i_ma50,28,0.416218,0.372341,0.012618,0.723802
2,t_max50,28,0.159089,0.239546,-0.473236,0.945777


Saved: E:\PJ_ADY\report\table_rq1c_load_per_cell.csv
Saved: E:\PJ_ADY\report\table_rq1c_load_summary.csv


In [33]:
# ============================================================
# RQ1D - PARTIAL SPEARMAN CORRELATION
#       controlling for cumulative throughput
# ============================================================

import numpy as np
import pandas as pd
from scipy.stats import rankdata, pearsonr
from sklearn.linear_model import LinearRegression
from pathlib import Path

features_rq1d = [
    "ah50",
    "i_ma50",
    "t_max50"
]

rq1d_rows = []

for cell, g in feat.groupby("cell"):

    for feature in features_rq1d:

        temp = (
            g[[feature, "soh", "cum_ah"]]
            .dropna()
            .copy()
        )

        # Convert all variables to ranks
        x_rank = rankdata(temp[feature])
        y_rank = rankdata(temp["soh"])
        c_rank = rankdata(temp["cum_ah"]).reshape(-1, 1)

        # Remove the monotonic effect of cumulative throughput
        model_x = LinearRegression().fit(
            c_rank,
            x_rank
        )

        model_y = LinearRegression().fit(
            c_rank,
            y_rank
        )

        x_residual = (
            x_rank
            - model_x.predict(c_rank)
        )

        y_residual = (
            y_rank
            - model_y.predict(c_rank)
        )

        partial_rho, p_value = pearsonr(
            x_residual,
            y_residual
        )

        rq1d_rows.append({
            "cell": cell,
            "feature": feature,
            "n_rows": len(temp),
            "partial_rho": partial_rho,
            "p_value": p_value
        })

rq1d = pd.DataFrame(
    rq1d_rows
)

rq1d_summary = (
    rq1d
    .groupby("feature")["partial_rho"]
    .agg(
        n_cells="count",
        median_partial_rho="median",
        mean_partial_rho="mean",
        min_partial_rho="min",
        max_partial_rho="max"
    )
    .reset_index()
)

rq1d.to_csv(
    "report/table_rq1d_partial_load_per_cell.csv",
    index=False
)

rq1d_summary.to_csv(
    "report/table_rq1d_partial_load_summary.csv",
    index=False
)

display(rq1d_summary)

print(
    "Saved:",
    Path(
        "report/table_rq1d_partial_load_per_cell.csv"
    ).resolve()
)

print(
    "Saved:",
    Path(
        "report/table_rq1d_partial_load_summary.csv"
    ).resolve()
)

,feature,n_cells,median_partial_rho,mean_partial_rho,min_partial_rho,max_partial_rho
0,ah50,28,-0.065597,-0.042571,-0.205468,0.219153
1,i_ma50,28,0.121690,0.122679,-0.011021,0.291263
2,t_max50,28,0.029429,0.035835,-0.105908,0.254660


Saved: E:\PJ_ADY\report\table_rq1d_partial_load_per_cell.csv
Saved: E:\PJ_ADY\report\table_rq1d_partial_load_summary.csv


## 6. RQ1 figures

The seven figures below are designed to answer different parts of RQ1 rather than repeat the same information.

All figures are saved at 300 dpi under `report/`.


In [34]:
# ============================================================
# FIGURE 1 - SOH DISTRIBUTION
# ============================================================

import matplotlib.pyplot as plt
from pathlib import Path

Path("report").mkdir(
    parents=True,
    exist_ok=True
)

fig, ax = plt.subplots(
    figsize=(7, 4)
)

ax.hist(
    feat["soh"],
    bins=30,
    edgecolor="black"
)

ax.axvline(
    0.8,
    linestyle="--",
    linewidth=1.5,
    label="EOL threshold (SOH = 0.8)"
)

ax.set_xlabel(
    "State of Health (SOH)"
)

ax.set_ylabel(
    "Number of operational steps"
)

ax.set_title(
    "Distribution of SOH across operational battery steps"
)

ax.spines[
    ["top", "right"]
].set_visible(False)

ax.legend()

fig.tight_layout()

out = Path(
    "report/fig_rq1_distribution.png"
)

fig.savefig(
    out,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print(
    "Saved:",
    out.resolve()
)

Saved: E:\PJ_ADY\report\fig_rq1_distribution.png


**Figure 1 — SOH distribution.**  
The feature table spans a broad health range, from roughly 0.33 to 0.99 SOH after requiring a full 50-step history. The dashed 0.8 line marks the EOL threshold used later in the project.


In [35]:
# ============================================================
# FIGURE 2 - REPRESENTATIVE DEGRADATION TRAJECTORIES
# ============================================================

representative_cells = [
    "RW11",
    "RW1",
    "RW26"
]

fig, ax = plt.subplots(
    figsize=(7, 4.5)
)

for cell in representative_cells:

    g = (
        q1[
            q1["cell"] == cell
        ]
        .sort_values("ah_bin")
    )

    ax.plot(
        g["ah_bin"],
        g["mean_soh"],
        marker="o",
        markersize=3,
        label=cell
    )

ax.axhline(
    0.8,
    linestyle="--",
    linewidth=1.2,
    label="SOH = 0.8"
)

ax.set_xlabel(
    "Cumulative operational throughput (Ah)"
)

ax.set_ylabel(
    "Mean SOH"
)

ax.set_title(
    "SOH declines with cumulative operational throughput"
)

ax.spines[
    ["top", "right"]
].set_visible(False)

ax.legend()

fig.tight_layout()

out = Path(
    "report/fig_rq1_timeseries.png"
)

fig.savefig(
    out,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print(
    "Saved:",
    out.resolve()
)

Saved: E:\PJ_ADY\report\fig_rq1_timeseries.png


**Figure 2 — Representative degradation trajectories.**  
Representative cells show the common downward relationship between accumulated throughput and SOH while also illustrating that cells reach the same health level at different accumulated throughput.


In [36]:
# ============================================================
# FIGURE 3 - ALL-CELL DEGRADATION TRAJECTORIES
# ============================================================

fig, ax = plt.subplots(
    figsize=(7.5, 5)
)

for cell, g in q1.groupby("cell"):

    g = g.sort_values(
        "ah_bin"
    )

    ax.plot(
        g["ah_bin"],
        g["mean_soh"],
        linewidth=0.9,
        alpha=0.65
    )

ax.axhline(
    0.8,
    linestyle="--",
    linewidth=1.3
)

ax.set_xlabel(
    "Cumulative operational throughput (Ah)"
)

ax.set_ylabel(
    "Mean SOH"
)

ax.set_title(
    "Degradation trajectories across all 28 cells"
)

ax.spines[
    ["top", "right"]
].set_visible(False)

fig.tight_layout()

out = Path(
    "report/fig_rq1_all_cells.png"
)

fig.savefig(
    out,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print(
    "Saved:",
    out.resolve()
)

Saved: E:\PJ_ADY\report\fig_rq1_all_cells.png


**Figure 3 — All-cell degradation trajectories.**  
All 28 cells exhibit declining SOH with increasing throughput, but their trajectories separate substantially. This cross-cell heterogeneity motivates the leave-cells-out evaluation in Notebook B.


In [37]:
# ============================================================
# FIGURE 4 - PER-CELL THROUGHPUT-SOH CORRELATION
# ============================================================

plot_rho = (
    rq1a
    .sort_values("spearman_rho")
    .copy()
)

fig, ax = plt.subplots(
    figsize=(8, 6)
)

ax.barh(
    plot_rho["cell"],
    plot_rho["spearman_rho"]
)

ax.axvline(
    0,
    linewidth=1
)

ax.set_xlabel(
    "Spearman correlation: cumulative throughput vs SOH"
)

ax.set_ylabel(
    "Battery cell"
)

ax.set_title(
    "Throughput is negatively associated with SOH in every cell"
)

ax.spines[
    ["top", "right"]
].set_visible(False)

fig.tight_layout()

out = Path(
    "report/fig_rq1_corr.png"
)

fig.savefig(
    out,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print(
    "Saved:",
    out.resolve()
)

Saved: E:\PJ_ADY\report\fig_rq1_corr.png


**Figure 4 — Per-cell throughput–SOH correlation.**  
Every cell has a strongly negative Spearman correlation. The observed range is approximately **−0.999 to −0.974**, showing that the monotonic direction is consistent across the dataset.


In [38]:
# ============================================================
# FIGURE 5 - DESCRIPTIVE DEGRADATION SLOPE BY CELL
# ============================================================

plot_slope = (
    rq1a
    .sort_values("soh_change_per_100ah")
    .copy()
)

fig, ax = plt.subplots(
    figsize=(8, 6)
)

ax.barh(
    plot_slope["cell"],
    plot_slope["soh_change_per_100ah"]
)

ax.axvline(
    0,
    linewidth=1
)

ax.set_xlabel(
    "Estimated SOH change per 100 Ah"
)

ax.set_ylabel(
    "Battery cell"
)

ax.set_title(
    "Battery cells exhibit heterogeneous degradation slopes"
)

ax.spines[
    ["top", "right"]
].set_visible(False)

fig.tight_layout()

out = Path(
    "report/fig_rq1_degradation_rate.png"
)

fig.savefig(
    out,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print(
    "Saved:",
    out.resolve()
)

Saved: E:\PJ_ADY\report\fig_rq1_degradation_rate.png


**Figure 5 — Descriptive degradation slopes.**  
The fitted linear slopes vary materially across cells, from roughly **−0.038 to −0.013 SOH per 100 Ah**. These slopes summarize whole-life trajectories and should be described as *descriptive linear degradation slopes*, not as exact physical degradation rates.


In [39]:
# ============================================================
# FIGURE 6 - RAW LOAD-HISTORY ASSOCIATIONS
# ============================================================

raw_corr = (
    rq1c_summary[
        [
            "feature",
            "median_rho"
        ]
    ]
    .copy()
)

fig, ax = plt.subplots(
    figsize=(6.5, 4)
)

bars = ax.bar(
    raw_corr["feature"],
    raw_corr["median_rho"]
)

ax.bar_label(
    bars,
    labels=[
        f"{x:.3f}"
        for x in raw_corr["median_rho"]
    ],
    padding=3
)

ax.axhline(
    0,
    linewidth=1
)

ax.set_ylabel(
    "Median cell-wise Spearman correlation"
)

ax.set_xlabel(
    "50-step load-history feature"
)

ax.set_title(
    "Raw load-history features are associated with SOH"
)

ax.spines[
    ["top", "right"]
].set_visible(False)

fig.tight_layout()

out = Path(
    "report/fig_rq1_load_raw.png"
)

fig.savefig(
    out,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print(
    "Saved:",
    out.resolve()
)

Saved: E:\PJ_ADY\report\fig_rq1_load_raw.png


**Figure 6 — Raw load-history associations.**  
The recent Ah and current summaries have moderate positive raw correlations with SOH, while the temperature-history correlation is weaker. These raw associations are potentially confounded and therefore are not interpreted causally.


In [40]:
# ============================================================
# FIGURE 7 - RAW VS THROUGHPUT-CONTROLLED ASSOCIATIONS
# ============================================================

compare = (
    rq1c_summary[
        [
            "feature",
            "median_rho"
        ]
    ]
    .merge(
        rq1d_summary[
            [
                "feature",
                "median_partial_rho"
            ]
        ],
        on="feature",
        how="inner"
    )
)

x = np.arange(
    len(compare)
)

width = 0.36

fig, ax = plt.subplots(
    figsize=(7, 4.5)
)

bars1 = ax.bar(
    x - width / 2,
    compare["median_rho"],
    width,
    label="Raw correlation"
)

bars2 = ax.bar(
    x + width / 2,
    compare["median_partial_rho"],
    width,
    label="Controlling cumulative throughput"
)

ax.bar_label(
    bars1,
    labels=[
        f"{v:.3f}"
        for v in compare["median_rho"]
    ],
    padding=2
)

ax.bar_label(
    bars2,
    labels=[
        f"{v:.3f}"
        for v in compare["median_partial_rho"]
    ],
    padding=2
)

ax.axhline(
    0,
    linewidth=1
)

ax.set_xticks(
    x,
    compare["feature"]
)

ax.set_ylabel(
    "Median cell-wise correlation"
)

ax.set_title(
    "Load-history associations weaken after controlling throughput"
)

ax.legend()

ax.spines[
    ["top", "right"]
].set_visible(False)

fig.tight_layout()

out = Path(
    "report/fig_rq1_partial_corr.png"
)

fig.savefig(
    out,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print(
    "Saved:",
    out.resolve()
)

Saved: E:\PJ_ADY\report\fig_rq1_partial_corr.png


**Figure 7 — Raw versus throughput-controlled associations.**  
After controlling cumulative throughput, all three median associations move much closer to zero. This is the key visual evidence that accumulated usage explains much of the apparent raw load-history relationship with SOH.


## 7. Final validation and Notebook B handoff

Before handoff, Notebook A verifies the feature table from several independent angles.

The final test suite checks:

- at least 30,000 rows and exactly 28 cells;
- unique `(cell, step)` keys;
- complete and valid SOH;
- exactly 50 operational history rows per feature row;
- absence of future reference-capacity columns in the feature table;
- finite core numerical features;
- removal of negative temperature sentinels;
- non-decreasing cumulative throughput within each cell.

A separate manual spot check recomputes `i_ma50`, `t_max50`, `ah50`, and `temp_valid_n50` from the 50 source rows for one deterministic RW10 example. The SQL and manual results must match.

Finally:

1. `feat.parquet` is exported;
2. `manifest.json` records schema, row count, cell list, step range, MD5 and SHA-256 hashes, SOH range, throughput range, and temperature missingness;
3. the Notebook B handoff simulation reloads both files and verifies the contract.

The verified handoff contains **548,010 rows, 17 columns, and 28 cells**.


In [41]:
# ============================================================
# FINAL TEST SUITE - NOTEBOOK A
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path

print("=" * 65)
print("NOTEBOOK A - FINAL VALIDATION")
print("=" * 65)

# TEST 1 - Basic size
assert len(feat) >= 30000, \
    f"FAIL: only {len(feat)} feature rows"

assert feat["cell"].nunique() == 28, \
    f"FAIL: expected 28 cells, got {feat['cell'].nunique()}"

print(
    f"[PASS] Dataset size: "
    f"{len(feat):,} rows, "
    f"{feat['cell'].nunique()} cells"
)

# TEST 2 - Unique key
duplicate_n = int(
    feat.duplicated(["cell", "step"]).sum()
)

assert duplicate_n == 0, \
    f"FAIL: {duplicate_n} duplicate cell-step records"

print("[PASS] cell + step is unique")

# TEST 3 - Target validity
assert feat["soh"].notna().all(), \
    "FAIL: SOH contains missing values"

assert (
    (feat["soh"] > 0) &
    (feat["soh"] <= 1)
).all(), \
    "FAIL: SOH outside (0, 1]"

print(
    "[PASS] SOH target valid: "
    f"{feat['soh'].min():.3f} to "
    f"{feat['soh'].max():.3f}"
)

# TEST 4 - Full 50-step history
assert (feat["history_n"] == 50).all(), \
    "FAIL: some rows do not have 50-step history"

assert (
    (feat["temp_valid_n50"] >= 0) &
    (feat["temp_valid_n50"] <= 50)
).all(), \
    "FAIL: temp_valid_n50 outside 0-50"

print(
    "[PASS] Every feature row has "
    "50 operational history steps"
)

# TEST 5 - Leakage-sensitive columns
forbidden_columns = {
    "capacity_ah",
    "capacity_initial_ah",
    "ref_step",
    "next_capacity",
    "future_capacity",
    "next_soh"
}

leakage_columns = forbidden_columns.intersection(
    set(feat.columns)
)

assert len(leakage_columns) == 0, \
    f"FAIL: potential leakage columns found: {leakage_columns}"

print(
    "[PASS] No future reference-capacity "
    "columns in feature table"
)

# TEST 6 - Numerical sanity
assert (feat["dur_s"] >= 0).all(), \
    "FAIL: negative duration"

assert (feat["ah"] >= 0).all(), \
    "FAIL: negative Ah"

assert np.isfinite(
    feat[
        [
            "dur_s",
            "v_start",
            "v_end",
            "i_mean",
            "ah",
            "cum_ah",
            "soh",
            "dv",
            "i_ma50",
            "ah50"
        ]
    ].to_numpy()
).all(), \
    "FAIL: unexpected non-finite core values"

print(
    "[PASS] Core numerical values are "
    "finite and physically consistent"
)

# TEST 7 - Temperature policy
negative_temperature = int(
    (feat["t_max"].dropna() < 0).sum()
)

assert negative_temperature == 0, \
    "FAIL: negative temperature sentinels remain"

temp_missing = int(
    feat["t_max"].isna().sum()
)

print(
    "[PASS] Temperature sentinel cleaning: "
    f"{temp_missing:,} t_max values intentionally marked missing"
)

# TEST 8 - cum_ah monotonic within cell
cum_decrease = (
    feat.sort_values(["cell", "step"])
        .groupby("cell")["cum_ah"]
        .diff()
        .dropna()
        .lt(-1e-10)
        .sum()
)

assert cum_decrease == 0, \
    f"FAIL: cumulative Ah decreases {cum_decrease} times"

print(
    "[PASS] Cumulative throughput is "
    "non-decreasing within every cell"
)

print("=" * 65)
print("ALL BASIC NOTEBOOK A TESTS PASSED")
print("=" * 65)

NOTEBOOK A - FINAL VALIDATION
[PASS] Dataset size: 548,010 rows, 28 cells
[PASS] cell + step is unique
[PASS] SOH target valid: 0.331 to 0.989
[PASS] Every feature row has 50 operational history steps
[PASS] No future reference-capacity columns in feature table
[PASS] Core numerical values are finite and physically consistent
[PASS] Temperature sentinel cleaning: 13,239 t_max values intentionally marked missing
[PASS] Cumulative throughput is non-decreasing within every cell
ALL BASIC NOTEBOOK A TESTS PASSED


In [42]:
# ============================================================
# MANUAL 50-STEP WINDOW SPOT CHECK
# ============================================================

# Pick a deterministic row well inside RW10
sample_row = (
    feat[
        feat["cell"] == "RW10"
    ]
    .sort_values("step")
    .iloc[500]
)

sample_cell = sample_row["cell"]
sample_step = sample_row["step"]

# Get exactly the 50 operational rows up to this step
manual_window = (
    rw[
        (rw["cell"] == sample_cell) &
        (rw["step"] <= sample_step)
    ]
    .sort_values("step")
    .tail(50)
)

assert len(manual_window) == 50

manual_i_ma50 = (
    manual_window["i_mean"]
    .abs()
    .mean()
)

manual_t_max50 = (
    manual_window["t_max"]
    .max()
)

manual_ah50 = (
    manual_window["ah"]
    .sum()
)

manual_temp_valid = (
    manual_window["t_max"]
    .notna()
    .sum()
)

print("Cell:", sample_cell)
print("Step:", sample_step)

print("\ni_ma50")
print("SQL   :", sample_row["i_ma50"])
print("Manual:", manual_i_ma50)

print("\nt_max50")
print("SQL   :", sample_row["t_max50"])
print("Manual:", manual_t_max50)

print("\nah50")
print("SQL   :", sample_row["ah50"])
print("Manual:", manual_ah50)

print("\ntemp_valid_n50")
print("SQL   :", sample_row["temp_valid_n50"])
print("Manual:", manual_temp_valid)

assert np.isclose(
    sample_row["i_ma50"],
    manual_i_ma50,
    rtol=1e-9,
    atol=1e-9
)

assert np.isclose(
    sample_row["t_max50"],
    manual_t_max50,
    rtol=1e-9,
    atol=1e-9,
    equal_nan=True
)

assert np.isclose(
    sample_row["ah50"],
    manual_ah50,
    rtol=1e-9,
    atol=1e-9
)

assert (
    sample_row["temp_valid_n50"]
    == manual_temp_valid
)

print(
    "\n[PASS] SQL rolling features match "
    "manual 50-step calculation"
)

Cell: RW10
Step: 1103

i_ma50
SQL   : 2.52203844746902
Manual: 2.52203844746902

t_max50
SQL   : 33.95863
Manual: 33.95863

ah50
SQL   : 7.404799545833333
Manual: 7.404799545833334

temp_valid_n50
SQL   : 50
Manual: 50

[PASS] SQL rolling features match manual 50-step calculation


In [43]:
# ============================================================
# FINAL FEATURE EXPORT
# ============================================================

from pathlib import Path

feature_path = Path(
    "data/processed/feat.parquet"
)

feat.to_parquet(
    feature_path,
    index=False
)

print(
    "Saved:",
    feature_path.resolve()
)

print(
    "Shape:",
    feat.shape
)

Saved: E:\PJ_ADY\data\processed\feat.parquet
Shape: (548010, 17)


In [44]:
# ============================================================
# CREATE HANDOFF MANIFEST FOR NOTEBOOK B
# ============================================================

import json
import hashlib
from datetime import datetime, timezone
from pathlib import Path

feature_path = Path(
    "data/processed/feat.parquet"
)

manifest_path = Path(
    "data/processed/manifest.json"
)

file_bytes = feature_path.read_bytes()

md5_hash = hashlib.md5(
    file_bytes
).hexdigest()

sha256_hash = hashlib.sha256(
    file_bytes
).hexdigest()

manifest = {
    "rows": int(len(feat)),
    "cols": list(feat.columns),
    "md5": md5_hash,
    "step_min": int(feat["step"].min()),
    "step_max": int(feat["step"].max()),

    "sha256": sha256_hash,
    "n_cells": int(feat["cell"].nunique()),
    "cells": sorted(
        feat["cell"].astype(str).unique().tolist()
    ),

    "soh_min": float(feat["soh"].min()),
    "soh_max": float(feat["soh"].max()),

    "cum_ah_min": float(
        feat["cum_ah"].min()
    ),
    "cum_ah_max": float(
        feat["cum_ah"].max()
    ),

    "history_window_steps": 50,

    "t_max_missing_n": int(
        feat["t_max"].isna().sum()
    ),

    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "author_role": "Notebook_A"
}

manifest_path.write_text(
    json.dumps(
        manifest,
        indent=2
    ),
    encoding="utf-8"
)

print(
    "Saved:",
    manifest_path.resolve()
)

print(
    json.dumps(
        manifest,
        indent=2
    )
)

Saved: E:\PJ_ADY\data\processed\manifest.json
{
  "rows": 548010,
  "cols": [
    "cell",
    "step",
    "type",
    "dur_s",
    "v_start",
    "v_end",
    "i_mean",
    "t_max",
    "ah",
    "cum_ah",
    "soh",
    "dv",
    "i_ma50",
    "t_max50",
    "ah50",
    "temp_valid_n50",
    "history_n"
  ],
  "md5": "00c8138e5deb3f253d76d6a381c4d8c5",
  "step_min": 98,
  "step_max": 113569,
  "sha256": "cba5e6f359578f2a5ab91f4033dafeaa753000ce28fe4b1e2668220d2da72760",
  "n_cells": 28,
  "cells": [
    "RW1",
    "RW10",
    "RW11",
    "RW12",
    "RW13",
    "RW14",
    "RW15",
    "RW16",
    "RW17",
    "RW18",
    "RW19",
    "RW2",
    "RW20",
    "RW21",
    "RW22",
    "RW23",
    "RW24",
    "RW25",
    "RW26",
    "RW27",
    "RW28",
    "RW3",
    "RW4",
    "RW5",
    "RW6",
    "RW7",
    "RW8",
    "RW9"
  ],
  "soh_min": 0.3309282894664863,
  "soh_max": 0.9894648415561227,
  "cum_ah_min": 5.789202122222222,
  "cum_ah_max": 3650.196579877778,
  "history_window_steps": 5

In [45]:
# ============================================================
# SIMULATE NOTEBOOK B HANDOFF CHECK
# ============================================================

import json
import hashlib
import pandas as pd

man = json.load(
    open(
        "data/processed/manifest.json",
        encoding="utf-8"
    )
)

current_md5 = hashlib.md5(
    open(
        "data/processed/feat.parquet",
        "rb"
    ).read()
).hexdigest()

handoff_feat = pd.read_parquet(
    "data/processed/feat.parquet"
)

assert current_md5 == man["md5"], \
    "FAIL: parquet hash differs from manifest"

assert len(handoff_feat) == man["rows"], \
    "FAIL: row count differs from manifest"

assert list(handoff_feat.columns) == man["cols"], \
    "FAIL: feature schema differs from manifest"

assert handoff_feat["cell"].nunique() == man["n_cells"], \
    "FAIL: cell count differs from manifest"

print("=" * 60)
print("NOTEBOOK B HANDOFF SIMULATION: PASS")
print("=" * 60)

print(
    f"Rows    : {man['rows']:,}"
)

print(
    f"Columns : {len(man['cols'])}"
)

print(
    f"Cells   : {man['n_cells']}"
)

print(
    f"Steps   : {man['step_min']} "
    f"to {man['step_max']}"
)

print(
    f"MD5     : {man['md5']}"
)

NOTEBOOK B HANDOFF SIMULATION: PASS
Rows    : 548,010
Columns : 17
Cells   : 28
Steps   : 98 to 113569
MD5     : 00c8138e5deb3f253d76d6a381c4d8c5


## 8. Supplementary temperature diagnostics

These cells are retained as supporting evidence rather than part of the main RQ1 argument.

The per-cell summary confirms that no negative temperatures remain after cleaning. It also shows substantial temperature differences between groups of cells, which is a warning that temperature may partly encode experimental campaign/protocol differences. Therefore, later SHAP or model importance must not be described as proof that temperature *causes* degradation.

The raw-array inspection provides an additional direct look at the original MATLAB temperature arrays behind the sentinel-value diagnosis.


In [46]:
# ============================================================
# TEMPERATURE BY CELL
# ============================================================

temp_by_cell = (
    rw.groupby('cell')
      .agg(
          rows=('t_max', 'size'),
          temp_mean=('t_max', 'mean'),
          temp_median=('t_max', 'median'),
          temp_min=('t_max', 'min'),
          temp_max=('t_max', 'max'),
          negative_temp_n=('t_max', lambda x: (x < 0).sum())
      )
      .reset_index()
)

temp_by_cell['negative_temp_pct'] = (
    temp_by_cell['negative_temp_n']
    / temp_by_cell['rows']
    * 100
)

display(temp_by_cell)

,cell,rows,temp_mean,temp_median,temp_min,temp_max,negative_temp_n,negative_temp_pct
0,RW1,7709,30.313496,30.47045,17.85481,44.65291,0,0.0
1,RW10,55267,34.279501,34.38987,20.93272,45.63619,0,0.0
2,RW11,54552,26.278024,26.08563,18.03504,38.71558,0,0.0
3,RW12,54463,26.960758,26.76226,18.16413,38.75309,0,0.0
4,RW13,27584,27.074781,27.53606,15.96744,36.91146,0,0.0
5,RW14,22819,26.563806,26.95281,15.56647,35.90049,0,0.0
6,RW15,23739,26.887015,27.21617,15.64627,37.01472,0,0.0
7,RW16,21117,27.311106,27.89759,16.02675,36.43322,0,0.0
8,RW17,15145,31.341680,31.47514,15.91736,49.62685,0,0.0
9,RW18,15701,31.230160,31.51799,0.19342,49.74214,0,0.0


In [47]:
# ============================================================
# INSPECT RAW TEMPERATURE ARRAYS
# ============================================================

from scipy.io import loadmat
import glob
import os
import numpy as np

f = sorted(
    glob.glob(
        'data/raw/rw_battery/**/RW10.mat',
        recursive=True
    )
)[0]

m = loadmat(
    f,
    simplify_cells=True
)['data']['step']

examples = []

for k, s in enumerate(m):
    T = np.atleast_1d(s['temperature'])

    if len(T) > 0:
        tmin = float(np.min(T))
        tmax = float(np.max(T))
        tmean = float(np.mean(T))

        if tmax < 0:
            examples.append({
                'step': k,
                'type': s['type'],
                'comment': str(s.get('comment', '')),
                'n_temp': len(T),
                'T_min': tmin,
                'T_mean': tmean,
                'T_max': tmax,
                'first_values': T[:5].tolist()
            })

    if len(examples) >= 10:
        break

display(pd.DataFrame(examples))

""


## 9. Reproducibility snapshot

The cells below record the software versions used in the verified `.venv` and regenerate the minimal `requirements.txt`.

The verified environment uses:

- Python 3.10.11
- NumPy 2.2.6
- pandas 2.3.3
- SciPy 1.15.3
- Matplotlib 3.10.0
- DuckDB 1.5.5
- scikit-learn 1.7.2
- PyArrow 25.0.1
- ydata-profiling 4.18.4

A clean run should be performed with the project root as the working directory and the `.venv` kernel selected in VS Code.


In [48]:
# ============================================================
# ENVIRONMENT VERSION SNAPSHOT
# ============================================================

import sys
import numpy
import pandas
import scipy
import matplotlib
import duckdb
import sklearn

print("Python       :", sys.version)
print("NumPy        :", numpy.__version__)
print("Pandas       :", pandas.__version__)
print("SciPy        :", scipy.__version__)
print("Matplotlib   :", matplotlib.__version__)
print("DuckDB       :", duckdb.__version__)
print("scikit-learn :", sklearn.__version__)

try:
    import pyarrow
    print("PyArrow      :", pyarrow.__version__)
except Exception as e:
    print("PyArrow      : not available")

try:
    import ydata_profiling
    print("ydata-profiling:", ydata_profiling.__version__)
except Exception as e:
    print("ydata-profiling: not available")

Python       : 3.10.11 (tags/v3.10.11:7d4cc5a, Apr  5 2023, 00:38:17) [MSC v.1929 64 bit (AMD64)]
NumPy        : 2.2.6
Pandas       : 2.3.3
SciPy        : 1.15.3
Matplotlib   : 3.10.0
DuckDB       : 1.5.5
scikit-learn : 1.7.2
PyArrow      : 25.0.1
ydata-profiling: 4.18.4


In [49]:
# ============================================================
# CREATE REQUIREMENTS.TXT
# ============================================================

from pathlib import Path

requirements = """\
numpy==2.2.6
pandas==2.3.3
scipy==1.15.3
matplotlib==3.10.0
duckdb==1.5.5
scikit-learn==1.7.2
pyarrow==25.0.1
ydata-profiling==4.18.4
ipykernel
"""

requirements_path = Path("requirements.txt")

requirements_path.write_text(
    requirements,
    encoding="utf-8"
)

print("Saved:", requirements_path.resolve())
print()
print(requirements_path.read_text())

Saved: E:\PJ_ADY\requirements.txt

numpy==2.2.6
pandas==2.3.3
scipy==1.15.3
matplotlib==3.10.0
duckdb==1.5.5
scikit-learn==1.7.2
pyarrow==25.0.1
ydata-profiling==4.18.4
ipykernel



## 10. Interpretation boundaries and known limitations

The following points should remain explicit in the report:

- **The SOH target is reference-test based.** Many operational rows share the same next-reference SOH until another reference test occurs. Row-level observations are therefore not statistically independent measurements of capacity.
- **Correlation is not causation.** RQ1 identifies monotonic associations and cross-cell patterns; it does not prove that any individual load variable causes degradation.
- **The linear slope is descriptive.** Real battery degradation can contain plateaus, knees, recovery-like measurement fluctuations, and accelerated late-life behavior.
- **Temperature missingness is deliberate.** Negative sentinel values are preserved as missing temperature measurements rather than converted to plausible-looking numbers in Notebook A.
- **Imputation belongs inside the training protocol.** Notebook B must fit imputation/scaling only on training cells.
- **Load history may still matter predictively.** Weak partial Spearman correlations do not rule out nonlinear interactions. This is why Notebook B should compare throughput-only, load-only, and combined feature sets.
- **Reference-test sensitivity is a useful extension.** The dataset contains repeated reference-discharge measurements. A later robustness analysis can compare first, second, and averaged reference capacities without changing the primary target definition used here.


## 11. Notebook A completion checklist

### Core data and reproducibility
- [x] NASA raw files found for all 28 cells
- [x] `data/processed/rw_battery.parquet`
- [x] `data/processed/feat.parquet`
- [x] `data/processed/manifest.json`
- [x] `sql/queries.sql`
- [x] `requirements.txt`
- [x] clean Restart → Run All completed successfully in the project `.venv`

### Data-quality evidence
- [x] automated profile
- [x] cleaning/preprocessing log
- [x] missing-value table
- [x] physical/range checks
- [x] per-cell SOH audit
- [x] temperature anomaly audit and post-clean validation

### RQ1 evidence
- [x] Q1–Q4 SQL outputs
- [x] RQ1A per-cell throughput correlations
- [x] RQ1B cross-cell Wilcoxon test
- [x] RQ1C raw load-history correlations
- [x] RQ1D throughput-controlled partial correlations
- [x] seven RQ1 figures

### Handoff
- [x] final test suite passes
- [x] manual 50-step rolling-window check passes
- [x] hash/schema handoff simulation passes

### Still to be completed outside this notebook
- [ ] final `README.md`
- [ ] final `.gitignore`
- [ ] archive old notebooks and stale report outputs
- [ ] cross-machine/team-member rerun evidence if required
- [ ] personal AI Audit Log / progress reporting


## 12. Handoff to Notebook B

Notebook A is complete when the clean run and handoff checks above pass.

Notebook B should begin by reading only:

```text
data/processed/feat.parquet
data/processed/manifest.json
```

and verifying the manifest before performing any split, imputation, scaling, tuning, or model fitting.

The next stage is the **cross-cell modeling pipeline**: leave-cells-out splits, five-model comparison, tuning, conformal uncertainty, EOL evaluation, feature ablation, SHAP/diagnosis, and deployment-cost analysis.
